In [36]:
import random
from IPython.display import Audio
import numpy as np
import matplotlib.pyplot as plt
import os
from glob import glob
import torchaudio
import random
import torch
def set_random_seed(seed: int):
    random.seed(seed)  # For Python random module
    np.random.seed(seed)  # For NumPy
    torch.manual_seed(seed)  # For PyTorch (CPU)
    torch.cuda.manual_seed(seed)  # For PyTorch (GPU)
    torch.backends.cudnn.deterministic = True  # Ensures deterministic behavior
    torch.backends.cudnn.benchmark = False  # Disables optimization for reproducibility

set_random_seed(999)
import warnings
warnings.filterwarnings("ignore")

In [37]:
def generate_random_msg(batch_size, msg_length, device):
    # random [0, 1], mapped to [-1, 1]
    return (
        torch.randint(0, 2, (batch_size, 1, msg_length), device=device).float() * 2
    ) - 1

def normalize_audio(y: torch.Tensor) -> torch.Tensor:
    """Normalize an audio tensor so its maximum absolute value is 1."""
    peak = torch.max(torch.abs(y))
    if peak.item() > 1e-8:
        y = y / peak
    return y

def save_spectrogram(signal, filepath, sample_rate=16000):
    plt.figure(figsize=(10, 4))
    plt.specgram(
        np.maximum(signal.cpu().detach().numpy(), 1e-10),
        Fs=sample_rate,
        NFFT=320,
        noverlap=160,
        window=np.hanning(320),
        cmap="magma",
        vmin=-100,
    )
    plt.colorbar(format="%+2.0f dB")
    plt.tight_layout()
    plt.savefig(filepath, format="png", bbox_inches="tight", pad_inches=0.0)
    plt.close()

def save_audio(signal, filepath="output.wav", sample_rate=16000):
    # signal shape: (1, length)
    torchaudio.save(filepath, signal.cpu(), sample_rate)
    return filepath

def splice_with_irrelevant(watermarked, irrelevant_paths, sample_rate=16000):
    """
    Replace 50% of each sample in a batch with random unrelated audio of same length.

    Args:
        watermarked (Tensor): (B, 1, L) batch of watermarked audio
        irrelevant_paths (List[str]): list of paths to unrelated .wav files
        sample_rate (int): target sample rate to match
    Returns:
        Tensor: spliced audio of same shape
    """
    B, C, L = watermarked.shape
    cut_len = L // 2
    mixed = watermarked.clone()

    for i in range(B):
        # Load a random unrelated audio
        rand_path = random.choice(irrelevant_paths)
        irre_audio, sr = torchaudio.load(rand_path)

        # Convert to mono and resample if needed
        if irre_audio.shape[0] > 1:
            irre_audio = irre_audio.mean(dim=0, keepdim=True)
        if sr != sample_rate:
            irre_audio = torchaudio.transforms.Resample(orig_freq=sr, new_freq=sample_rate)(irre_audio)

        # Trim or pad irrelevant audio to at least cut_len
        if irre_audio.shape[1] < cut_len:
            pad_len = cut_len - irre_audio.shape[1]
            irre_audio = torch.nn.functional.pad(irre_audio, (0, pad_len))
        irre_audio = irre_audio[:, :cut_len]

        # Pick splice position
        start = random.randint(0, L - cut_len)
        end = start + cut_len

        # Replace middle 50% with irrelevant audio
        mixed[i, :, start:end] = irre_audio

    return mixed


import random
import torch
import torchaudio

def splice_with_irrelevant_scattered(watermarked, irrelevant_paths, sample_rate=16000, num_segments=10):
    """
    Replaces 50% of each audio in a batch with unrelated audio, scattered over multiple random segments.

    Args:
        watermarked (Tensor): (B, 1, L) batch of watermarked audio
        irrelevant_paths (List[str]): list of paths to unrelated .wav files
        sample_rate (int): sample rate of the audio
        num_segments (int): number of scattered segments to use (default: 10)

    Returns:
        Tensor: Corrupted audio of shape (B, 1, L)
    """
    B, C, L = watermarked.shape
    target_total = L // 2  # Replace 50% of audio
    mixed = watermarked.clone()

    for i in range(B):
        # Randomly divide 50% length into `num_segments` random positive integers
        # Ensures segments have variable lengths but sum to target_total
        segment_lengths = torch.randint(low=1, high=L // (2 * num_segments), size=(num_segments,))
        scale = target_total / segment_lengths.sum().item()
        segment_lengths = (segment_lengths.float() * scale).long()
        segment_lengths[-1] += target_total - segment_lengths.sum()  # Adjust rounding error

        # Keep track of inserted regions to avoid overlaps
        used_ranges = []
        positions = []

        for seg_len in segment_lengths:
            for _ in range(20):  # Try 20 times to find non-overlapping segment
                start = random.randint(0, L - seg_len)
                end = start + seg_len
                if all(end <= s or start >= e for s, e in used_ranges):
                    used_ranges.append((start, end))
                    positions.append((start, end, seg_len))
                    break

        for start, end, seg_len in positions:
            rand_path = random.choice(irrelevant_paths)
            irre_audio, sr = torchaudio.load(rand_path)

            # Convert to mono
            if irre_audio.shape[0] > 1:
                irre_audio = irre_audio.mean(dim=0, keepdim=True)

            # Resample if needed
            if sr != sample_rate:
                irre_audio = torchaudio.transforms.Resample(orig_freq=sr, new_freq=sample_rate)(irre_audio)

            # Ensure it's long enough
            if irre_audio.shape[1] < seg_len:
                irre_audio = torch.nn.functional.pad(irre_audio, (0, seg_len - irre_audio.shape[1]))

            irre_audio = irre_audio[:, :seg_len]
            assert irre_audio.shape == (1, seg_len), f"Shape mismatch: got {irre_audio.shape}, expected (1, {seg_len})"

            mixed[i, :, start:end] = irre_audio

    return mixed


# MSE + Loudness Model

## Load Model Architecture

In [38]:
import torch.nn as nn
from torch.nn import LeakyReLU
from watermarking_model.model.blocks import FCBlock, Conv2Encoder, WatermarkEmbedder, WatermarkExtracter, ReluBlock
from distortions.frequency import TacotronSTFT, fixed_STFT, tacotron_mel
from silero_vad import load_silero_vad
import yaml
from dataset.data import WavDataset as MyDataset
from dataset.data import collate_fn
from torch.utils.data import DataLoader
import torch
import torchaudio
from rich.progress import track
from torch.nn.functional import mse_loss

# Optional: set up a small constant
EPS = 1e-9

class Encoder(nn.Module):
    def __init__(self, process_config, model_config, train_config, msg_length):
        super(Encoder, self).__init__()
        self.name = "conv2"
        self.win_dim = int((process_config["mel"]["n_fft"] / 2) + 1)
        self.add_carrier_noise = False
        self.block = model_config["conv2"]["block"]
        self.layers_CE = model_config["conv2"]["layers_CE"]
        self.EM_input_dim = model_config["conv2"]["hidden_dim"] + 3
        self.layers_EM = model_config["conv2"]["layers_EM"]
        self.n_fft = process_config["mel"]["n_fft"]
        self.hop_length = process_config["mel"]["hop_length"]
        self.win_length = process_config["mel"]["win_length"]
        self.sampling_rate = process_config["audio"]["or_sample_rate"]
        self.delay_amt = int((train_config["watermark"]["delay_amt_second"]*self.sampling_rate) // self.hop_length + 1)
        self.future_amt = int((train_config["watermark"]["future_amt_second"]*self.sampling_rate) // self.hop_length + 1)
        self.delay = True
        self.power = 1.0
        self.vad = load_silero_vad()
        self.vad_threshold = 0.50

        self.vocoder_step = model_config["structure"]["vocoder_step"]
        #MLP for the input wm
        self.msg_linear_in = FCBlock(msg_length, self.win_dim//2, activation=LeakyReLU(inplace=True))

        #stft transform
        self.stft = fixed_STFT(process_config["mel"]["n_fft"], process_config["mel"]["hop_length"], process_config["mel"]["win_length"])

        self.ENc = Conv2Encoder(input_channel=2, hidden_dim = model_config["conv2"]["hidden_dim"], block=self.block, n_layers=self.layers_CE)

        self.EM = WatermarkEmbedder(input_channel=self.EM_input_dim, hidden_dim = model_config["conv2"]["hidden_dim"], block=self.block, n_layers=self.layers_EM)

    def pad_w_zero_stft(self, input_stft, watermark_stft, voice_prefilling):
        """
        Pad the watermarked stft output with zeros on the left + right,
        respecting future_amt and chunk-based offsets.
        """
        chunk_size = voice_prefilling if not self.delay else (voice_prefilling + self.delay_amt)

        zeros_right_len = input_stft.shape[3] - watermark_stft.shape[3] - chunk_size
        if zeros_right_len < 0:
            # Edge case: won't happen if chunking logic is correct, but just to be safe
            zeros_right_len = 0

        zeros_left = torch.zeros_like(input_stft[:, :, :, :chunk_size])
        zeros_right = torch.zeros_like(input_stft[:, :, :, :zeros_right_len])

        actual_watermark = torch.cat([zeros_left, watermark_stft, zeros_right], dim=3) + EPS
        return actual_watermark

    def forward(self, x, msg, global_step):
        num_samples = x.shape[-1]
        _, _, stft_result = self.stft.transform(x)
        # Evaluate how many chunks we can process
        # 2s input + 0.5s calculation delay
        # 2.00*16000 = 32000
        # 32800 // hop_length + 1 = 201 center=True
        # 0.5s*16000 = 8000
        # 8000 // hop_length + 1 = 51 center=True
        voice_prefilling = int((2.00*self.sampling_rate)//self.hop_length + 1)
        # Predict future 0.5s watermark
        # 0.5*16000 = 8000
        # 8000 // hop_length + 1 =51
        max_start = stft_result.shape[-1] - (voice_prefilling + self.delay_amt)
        if int(max_start / self.delay_amt) <= 0:
            return None  # Not enough frames for a chunk

        list_of_watermarks = []
        for i in range(int((stft_result.shape[-1] - (voice_prefilling + self.delay_amt)) / self.future_amt)):
            carrier_encoded = self.ENc(stft_result[:, :, :, i * self.future_amt:voice_prefilling + i * self.future_amt])
            # torch.Size([B, 1, 81])
            # torch.Size([B, 81, 1])
            # torch.Size([B, 1, 81, 1])
            # torch.Size([B, 1, 162, 201])
            watermark_encoded = self.msg_linear_in(msg).transpose(1, 2).unsqueeze(1).repeat(1, 1, 2,
                                                                                            carrier_encoded.shape[3])
            concatenated_feature = torch.cat((carrier_encoded, stft_result[:, :, :,
                                                               i*self.future_amt:voice_prefilling + i*self.future_amt], watermark_encoded), dim=1)
            # [B, 2, bins, length]
            # Embed the watermark
            carrier_watermarked = self.EM(concatenated_feature)
            # Append both the watermark chunk and the pilot segment
            list_of_watermarks.append(carrier_watermarked)

        if len(list_of_watermarks) > 0:
            watermark = torch.cat(list_of_watermarks, dim=-1)
            all_watermark_stft = self.pad_w_zero_stft(
                stft_result, watermark, voice_prefilling
            )
            del list_of_watermarks
            mask=stft_result!=0
            all_watermark_stft = all_watermark_stft*mask + 0.0000001

            self.stft.num_samples = num_samples

            # Recompute magnitude & phase
            real_part = all_watermark_stft[:, 0, :, :]
            imag_part = all_watermark_stft[:, 1, :, :]
            spect = torch.sqrt(real_part ** 2 + imag_part ** 2)
            phase = torch.atan2(imag_part, real_part)

            y = self.stft.inverse(spect, phase).squeeze(1)
            del spect, phase, real_part, imag_part

            # with torch.no_grad():
            #     # Get chunk-level speech probabilities for the batch.
            #     # The output shape should be [batch, num_chunks]
            #     batch_chunk_probs = self.vad.audio_forward(x, sr=self.sampling_rate)
            #
            # # Threshold the probabilities to obtain a binary mask per chunk.
            # batch_chunk_mask = (batch_chunk_probs > self.vad_threshold).float()
            #
            # # Upsample the chunk-level mask to a sample-level mask.
            # # Each chunk's decision is repeated for chunk_size samples.
            # sample_masks = torch.repeat_interleave(batch_chunk_mask, 512, dim=1).to(y.device)
            #
            # # Since the upsampled mask might be longer than the actual audio length,
            # # slice the mask to match the original number of samples.
            # sample_length = x.shape[-1]
            # sample_masks = sample_masks[:, :sample_length]
            #
            # # Apply the mask to the original audio to zero out non-speech regions.
            # masked_y = y * sample_masks
            return y, all_watermark_stft
        else:
            print("Not enough watermarking!!!!")
            return None


class Decoder(nn.Module):
    def __init__(self, process_config, model_config, train_config, msg_length):
        super(Decoder, self).__init__()
        self.robust = model_config["robust"]
        # if self.robust:
        #     self.dl = distortion(process_config, train_config)
        self.mel_transform = TacotronSTFT(filter_length=process_config["mel"]["n_fft"], hop_length=process_config["mel"]["hop_length"], win_length=process_config["mel"]["win_length"])
        # self.vocoder = get_vocoder(device)
        self.vocoder_step = model_config["structure"]["vocoder_step"]
        self.win_dim = int((process_config["mel"]["n_fft"] / 2) + 1)
        self.hop_length = process_config["mel"]["hop_length"]
        self.block = model_config["conv2"]["block"]
        self.EX = WatermarkExtracter(input_channel=2, hidden_dim=model_config["conv2"]["hidden_dim"], block=self.block)
        self.stft = fixed_STFT(process_config["mel"]["n_fft"], process_config["mel"]["hop_length"], process_config["mel"]["win_length"])
        self.msg_linear_out = FCBlock(self.win_dim, msg_length, activation=LeakyReLU(inplace=True))

    def forward(self, y, global_step):
        y_identity = y
        # if global_step > self.vocoder_step:
        #     y_mel = self.mel_transform.mel_spectrogram(y.squeeze(1))
        #     # y = self.vocoder(y_mel)
        #     y_d = (self.mel_transform.griffin_lim(magnitudes=y_mel)).unsqueeze(1)
        # else:
        #     y_d = y
        y_d = y

        spect, phase, stft_result = self.stft.transform(y_d.squeeze(1))
        extracted_wm = self.EX(stft_result).squeeze(1)  # (B, win_dim, length)
        # # Explicitly split the 162-dim vector into two halves of 81-dim each
        # low, high = extracted_wm.chunk(2, dim=1)  # each has shape [B, win_dim / 2, length]
        # low_msg = torch.mean(low, dim=2, keepdim=True).transpose(1,2)
        # high_msg = torch.mean(high, dim=2, keepdim=True).transpose(1, 2)
        # msg_avg = (low_msg + high_msg) / 2  # Average the two halves -> shape: [B, 1, 81]
        msg = torch.mean(extracted_wm, dim=2, keepdim=True).transpose(1,2)
        msg = self.msg_linear_out(msg)
        # msg = self.msg_linear_out(msg_avg)

        _, _, stft_result_identity = self.stft.transform(y_identity)
        extracted_wm_identity = self.EX(stft_result_identity).squeeze(1)
        # low_identity, high_identity = extracted_wm_identity.chunk(2, dim=1)  # each has shape [B, win_dim / 2, length]
        # low_msg_identity = torch.mean(low_identity, dim=2, keepdim=True).transpose(1, 2)
        # high_msg_identity = torch.mean(high_identity, dim=2, keepdim=True).transpose(1, 2)
        # msg_avg_identity = (low_msg_identity + high_msg_identity) / 2  # Average the two halves -> shape: [B, 1, 81]
        msg_identity = torch.mean(extracted_wm_identity,dim=2, keepdim=True).transpose(1,2)
        msg_identity = self.msg_linear_out(msg_identity)
        # msg_identity = self.msg_linear_out(msg_avg_identity)
        del stft_result, stft_result_identity, extracted_wm, extracted_wm_identity
        return msg, msg_identity

class Discriminator(nn.Module):
    def __init__(self, process_config):
        super(Discriminator, self).__init__()
        self.conv = nn.Sequential(
                ReluBlock(2,16,3,1,1),
                ReluBlock(16,32,3,1,1),
                ReluBlock(32,64,3,1,1),
                nn.AdaptiveAvgPool2d(output_size=(1, 1))
                )
        self.linear = nn.Linear(64,1)
        self.stft = fixed_STFT(process_config["mel"]["n_fft"], process_config["mel"]["hop_length"], process_config["mel"]["win_length"])

    def forward(self, x):
        _, _, stft_result = self.stft.transform(x)
        x = self.conv(stft_result)
        x = x.squeeze(2).squeeze(2)
        x = self.linear(x)
        return x

In [39]:
process_config = yaml.load(open("./config/process.yaml", "r"), Loader=yaml.FullLoader)
model_config = yaml.load(open("./config/model.yaml", "r"), Loader=yaml.FullLoader)
train_config = yaml.load(open("./config/train.yaml", "r"), Loader=yaml.FullLoader)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

encoder = Encoder(process_config, model_config, train_config, train_config["watermark"]["length"]).to(device)
decoder = Decoder(process_config, model_config, train_config, train_config["watermark"]["length"]).to(device)
discriminator = Discriminator(process_config).to(device)
checkpoint_path = "ablation/original_baseline-conv2_ep_30_2025-05-01_05_51_48.pth.tar"
checkpoint = torch.load(checkpoint_path, map_location=device)
encoder.load_state_dict(checkpoint["encoder"])
decoder.load_state_dict(checkpoint["decoder"])

<All keys matched successfully>

## Single Inference

In [35]:
with torch.inference_mode():
    encoder.eval()
    decoder.eval()
    msg = generate_random_msg(1, train_config["watermark"]["length"], device)
    # wav, sr = torchaudio.load("ablation/260-123440-0016.wav")
    # wav, sr = torchaudio.load("ablation/LJ001-0052.wav")
    wav, sr = torchaudio.load("ablation/LJ001-0053.wav")
    wav = wav.to(device) # (batch, length)
    watermark, carrier_wateramrked = encoder(wav, msg, 1) # (batch, 1, length)
    y_wm = wav + watermark
    decoded = decoder(y_wm, 1)
    decoder_acc = [((decoded[0] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item(),
                           ((decoded[1] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item()]
    print("Accuracy:[{:.8f},{:.8f}]".format(*decoder_acc))
    print("original msg:", msg)
    print("decoded msg:", decoded[0])

    y_norm = normalize_audio(wav.cpu().detach())
    wm_norm = normalize_audio(watermark[0].cpu().detach())
    y_wm_norm = normalize_audio(y_wm[0].cpu().detach())

    save_spectrogram(y_norm.squeeze(0), "ablation/orig_spectrogram.png")
    save_spectrogram(wm_norm, "ablation/MSE_loudness_baseline_watermark_spectrogram.png")
    save_spectrogram(y_wm_norm, "ablation/MSE_loudness_watermarked_spectrogram.png")

    save_audio(wav, "ablation/original_audio.wav", sample_rate=sr)
    save_audio(watermark, "ablation/MSE_loudness_watermark_audio.wav", sample_rate=sr)
    save_audio(y_wm, "ablation/MSE_loudness_watermarked_audio.wav", sample_rate=sr)

Accuracy:[1.00000000,1.00000000]
original msg: tensor([[[ 1.,  1., -1.,  1., -1., -1.,  1.,  1., -1., -1.]]], device='cuda:0')
decoded msg: tensor([[[ 1.0077,  0.8181, -0.0129,  1.0974, -0.0106, -0.0168,  0.8188,
           1.2321, -0.0101, -0.0169]]], device='cuda:0')


### Original Audio

In [6]:
Audio(wav.cpu(), rate=sr)

### Watermarked Audio

In [7]:
Audio(y_wm.cpu(), rate=sr)

## Batch Inference

### LibriSpeech

In [ ]:
train_config["path"]["raw_path"] = "/home/yizhu/Data/LibriSpeech_wav"
dev_audios = MyDataset(
    process_config=process_config, train_config=train_config, flag="test"
)
dev_audios_loader = DataLoader(
        dev_audios,
        batch_size=2,
        shuffle=False,
        collate_fn=collate_fn,
        pin_memory=True,
        num_workers=20,
        persistent_workers=True,
    )

In [ ]:
with torch.inference_mode():
    encoder.eval()
    decoder.eval()
    discriminator.eval()
    test_avg_acc = [0, 0]
    test_avg_snr = 0
    count = 0
    for sample in track(dev_audios_loader):
        count += 1
        b = sample["matrix"].shape[0]
        # ---------------- build watermark
        wav_matrix = sample["matrix"].to(device)
        msg = generate_random_msg(wav_matrix.size(0), train_config["watermark"]["length"], device)
        watermark, carrier_wateramrked = encoder(wav_matrix, msg, 1)
        y_wm = wav_matrix + watermark
        decoded = decoder(y_wm, 1)
        decoder_acc = [((decoded[0] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item(),
                           ((decoded[1] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item()]
        zero_tensor = torch.zeros(wav_matrix.shape).to(device)
        snr = 10 * torch.log10(
            mse_loss(wav_matrix.detach(), zero_tensor) / mse_loss(wav_matrix.detach(), y_wm.detach()))

        test_avg_snr += snr
        test_avg_acc[0] += decoder_acc[0]
        test_avg_acc[1] += decoder_acc[1]

    test_avg_acc[0] /= count
    test_avg_acc[1] /= count
    test_avg_snr /= count
    print("Test Average SNR:", test_avg_snr)
    print("Test Average Accuracy:", test_avg_acc[0])


In [ ]:
count

### LJSpeech

In [ ]:
train_config["path"]["raw_path"] = "/home/yizhu/Data/LJSpeech-1.1"
dev_audios = MyDataset(
    process_config=process_config, train_config=train_config, flag="test"
)
dev_audios_loader = DataLoader(
        dev_audios,
        batch_size=2,
        shuffle=False,
        collate_fn=collate_fn,
        pin_memory=True,
        num_workers=20,
        persistent_workers=True,
    )

In [ ]:
with torch.inference_mode():
    encoder.eval()
    decoder.eval()
    discriminator.eval()
    test_avg_acc = [0, 0]
    test_avg_snr = 0
    count = 0
    for sample in track(dev_audios_loader):
        count += 1
        b = sample["matrix"].shape[0]
        # ---------------- build watermark
        wav_matrix = sample["matrix"].to(device)
        msg = generate_random_msg(wav_matrix.size(0), train_config["watermark"]["length"], device)
        watermark, carrier_wateramrked = encoder(wav_matrix, msg, 1)
        y_wm = wav_matrix + watermark
        decoded = decoder(y_wm, 1)
        decoder_acc = [((decoded[0] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item(),
                           ((decoded[1] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item()]
        zero_tensor = torch.zeros(wav_matrix.shape).to(device)
        snr = 10 * torch.log10(
            mse_loss(wav_matrix.detach(), zero_tensor) / mse_loss(wav_matrix.detach(), y_wm_aug.detach()))

        test_avg_snr += snr
        test_avg_acc[0] += decoder_acc[0]
        test_avg_acc[1] += decoder_acc[1]

    test_avg_acc[0] /= count
    test_avg_acc[1] /= count
    test_avg_snr /= count
    print("Test Average SNR:", test_avg_snr)
    print("Test Average Accuracy:", test_avg_acc[0])

## Background Noise

### 3-30 snr db

#### LibriSpeech

In [40]:
train_config["path"]["raw_path"] = "/home/yizhu/Data/LibriSpeech_wav"
dev_audios = MyDataset(
    process_config=process_config, train_config=train_config, flag="test"
)
dev_audios_loader = DataLoader(
        dev_audios,
        batch_size=2,
        shuffle=False,
        collate_fn=collate_fn,
        pin_memory=True,
        num_workers=20,
        persistent_workers=True,
    )

In [41]:
from audiomentations import AddBackgroundNoise, PolarityInversion

# Fix randomness
random.seed(42)
np.random.seed(42)

transform = AddBackgroundNoise(
    sounds_path="ablation/ESC-50-master/audio",  # Replace with real path
    min_snr_db=3.0,
    max_snr_db=30.0,
    noise_transform=PolarityInversion(),
    p=1.0
)

with torch.inference_mode():
    encoder.eval()
    decoder.eval()
    discriminator.eval()
    test_avg_acc = [0, 0]
    test_avg_snr = 0
    count = 0
    for sample in track(dev_audios_loader):
        count += 1
        b = sample["matrix"].shape[0]
        # ---------------- build watermark
        wav_matrix = sample["matrix"].to(device)
        # Fixed message for testing (same for all in batch)
        msg = torch.tensor([-1.,  1.,  1., -1., -1.,  1., -1.,  1., -1., -1.], dtype=torch.float32)
        msg = msg.unsqueeze(0).repeat(b, 1, 1).to(device)

        # Watermark embedding
        watermark, carrier_wateramrked = encoder(wav_matrix, msg, 1)
        y_wm = wav_matrix + watermark

        # Convert each sample to NumPy, apply random noise, convert back
        # Convert each sample to NumPy, apply random noise, convert back
        augmented_waveforms = []
        for i in range(y_wm.shape[0]):
            waveform_np = y_wm[i].detach().cpu().numpy()
            if waveform_np.ndim > 1:
                waveform_np = waveform_np[0]  # Select first channel if stereo
            augmented_np = transform(samples=waveform_np, sample_rate=16000)
            augmented_waveforms.append(torch.tensor(augmented_np).unsqueeze(0))  # Add (1, L)

        y_wm_aug = torch.cat(augmented_waveforms, dim=0).to(device)  # Shape: (B, L)

        decoded = decoder(y_wm_aug, 1)
        decoder_acc = [((decoded[0] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item(),
                           ((decoded[1] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item()]
        zero_tensor = torch.zeros(wav_matrix.shape).to(device)
        snr = 10 * torch.log10(
            mse_loss(wav_matrix.detach(), zero_tensor) / mse_loss(wav_matrix.detach(), y_wm_aug.detach()))

        test_avg_snr += snr
        test_avg_acc[0] += decoder_acc[0]
        test_avg_acc[1] += decoder_acc[1]

    test_avg_acc[0] /= count
    test_avg_acc[1] /= count
    test_avg_snr /= count
    print("Test Average SNR:", test_avg_snr)
    print("Test Average Accuracy:", test_avg_acc[0])

Test Average SNR: tensor(13.2433, device='cuda:0')
Test Average Accuracy: 0.8722222381167941


In [42]:
Audio(y_wm_aug[0].cpu(), rate=sr)

#### LJSpeech

In [43]:
train_config["path"]["raw_path"] = "/home/yizhu/Data/LJSpeech-1.1"
dev_audios = MyDataset(
    process_config=process_config, train_config=train_config, flag="test"
)
dev_audios_loader = DataLoader(
        dev_audios,
        batch_size=2,
        shuffle=False,
        collate_fn=collate_fn,
        pin_memory=True,
        num_workers=20,
        persistent_workers=True,
    )

In [44]:
from audiomentations import AddBackgroundNoise, PolarityInversion

# Fix randomness
random.seed(42)
np.random.seed(42)

transform = AddBackgroundNoise(
    sounds_path="ablation/ESC-50-master/audio",  # Replace with real path
    min_snr_db=3.0,
    max_snr_db=30.0,
    noise_transform=PolarityInversion(),
    p=1.0
)

with torch.inference_mode():
    encoder.eval()
    decoder.eval()
    discriminator.eval()
    test_avg_acc = [0, 0]
    test_avg_snr = 0
    count = 0
    for sample in track(dev_audios_loader):
        count += 1
        b = sample["matrix"].shape[0]
        # ---------------- build watermark
        wav_matrix = sample["matrix"].to(device)
        # Fixed message for testing (same for all in batch)
        msg = torch.tensor([-1.,  1.,  1., -1., -1.,  1., -1.,  1., -1., -1.], dtype=torch.float32)
        msg = msg.unsqueeze(0).repeat(b, 1, 1).to(device)

        # Watermark embedding
        watermark, carrier_wateramrked = encoder(wav_matrix, msg, 1)
        y_wm = wav_matrix + watermark

        # Convert each sample to NumPy, apply random noise, convert back
        # Convert each sample to NumPy, apply random noise, convert back
        augmented_waveforms = []
        for i in range(y_wm.shape[0]):
            waveform_np = y_wm[i].detach().cpu().numpy()
            if waveform_np.ndim > 1:
                waveform_np = waveform_np[0]  # Select first channel if stereo
            augmented_np = transform(samples=waveform_np, sample_rate=16000)
            augmented_waveforms.append(torch.tensor(augmented_np).unsqueeze(0))  # Add (1, L)

        y_wm_aug = torch.cat(augmented_waveforms, dim=0).to(device)  # Shape: (B, L)

        decoded = decoder(y_wm_aug, 1)
        decoder_acc = [((decoded[0] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item(),
                           ((decoded[1] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item()]
        zero_tensor = torch.zeros(wav_matrix.shape).to(device)
        snr = 10 * torch.log10(
            mse_loss(wav_matrix.detach(), zero_tensor) / mse_loss(wav_matrix.detach(), y_wm_aug.detach()))

        test_avg_snr += snr
        test_avg_acc[0] += decoder_acc[0]
        test_avg_acc[1] += decoder_acc[1]

    test_avg_acc[0] /= count
    test_avg_acc[1] /= count
    test_avg_snr /= count
    print("Test Average SNR:", test_avg_snr)
    print("Test Average Accuracy:", test_avg_acc[0])

Test Average SNR: tensor(12.8808, device='cuda:0')
Test Average Accuracy: 0.9749999940395355


In [45]:
Audio(y_wm_aug[0].cpu(), rate=sr)

### 5 snr db

#### LibriSpeech

In [46]:
train_config["path"]["raw_path"] = "/home/yizhu/Data/LibriSpeech_wav"
dev_audios = MyDataset(
    process_config=process_config, train_config=train_config, flag="test"
)
dev_audios_loader = DataLoader(
        dev_audios,
        batch_size=2,
        shuffle=False,
        collate_fn=collate_fn,
        pin_memory=True,
        num_workers=20,
        persistent_workers=True,
    )

In [47]:
from audiomentations import AddBackgroundNoise, PolarityInversion

# Fix randomness
random.seed(42)
np.random.seed(42)

transform = AddBackgroundNoise(
    sounds_path="ablation/ESC-50-master/audio",  # Replace with real path
    min_snr_db=5.0,
    max_snr_db=5.0,
    noise_transform=PolarityInversion(),
    p=1.0
)

with torch.inference_mode():
    encoder.eval()
    decoder.eval()
    discriminator.eval()
    test_avg_acc = [0, 0]
    test_avg_snr = 0
    count = 0
    for sample in track(dev_audios_loader):
        count += 1
        b = sample["matrix"].shape[0]
        # ---------------- build watermark
        wav_matrix = sample["matrix"].to(device)
        # Fixed message for testing (same for all in batch)
        msg = torch.tensor([-1.,  1.,  1., -1., -1.,  1., -1.,  1., -1., -1.], dtype=torch.float32)
        msg = msg.unsqueeze(0).repeat(b, 1, 1).to(device)

        # Watermark embedding
        watermark, carrier_wateramrked = encoder(wav_matrix, msg, 1)
        y_wm = wav_matrix + watermark

        # Convert each sample to NumPy, apply random noise, convert back
        # Convert each sample to NumPy, apply random noise, convert back
        augmented_waveforms = []
        for i in range(y_wm.shape[0]):
            waveform_np = y_wm[i].detach().cpu().numpy()
            if waveform_np.ndim > 1:
                waveform_np = waveform_np[0]  # Select first channel if stereo
            augmented_np = transform(samples=waveform_np, sample_rate=16000)
            augmented_waveforms.append(torch.tensor(augmented_np).unsqueeze(0))  # Add (1, L)

        y_wm_aug = torch.cat(augmented_waveforms, dim=0).to(device)  # Shape: (B, L)

        decoded = decoder(y_wm_aug, 1)
        decoder_acc = [((decoded[0] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item(),
                           ((decoded[1] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item()]
        zero_tensor = torch.zeros(wav_matrix.shape).to(device)
        snr = 10 * torch.log10(
            mse_loss(wav_matrix.detach(), zero_tensor) / mse_loss(wav_matrix.detach(), y_wm_aug.detach()))

        test_avg_snr += snr
        test_avg_acc[0] += decoder_acc[0]
        test_avg_acc[1] += decoder_acc[1]

    test_avg_acc[0] /= count
    test_avg_acc[1] /= count
    test_avg_snr /= count
    print("Test Average SNR:", test_avg_snr)
    print("Test Average Accuracy:", test_avg_acc[0])

Test Average SNR: tensor(4.9489, device='cuda:0')
Test Average Accuracy: 0.8333333399560716


In [48]:
Audio(y_wm_aug[0].cpu(), rate=sr)

In [49]:
train_config["path"]["raw_path"] = "/home/yizhu/Data/LJSpeech-1.1"
dev_audios = MyDataset(
    process_config=process_config, train_config=train_config, flag="test"
)
dev_audios_loader = DataLoader(
        dev_audios,
        batch_size=2,
        shuffle=False,
        collate_fn=collate_fn,
        pin_memory=True,
        num_workers=20,
        persistent_workers=True,
    )

In [50]:
from audiomentations import AddBackgroundNoise, PolarityInversion

# Fix randomness
random.seed(42)
np.random.seed(42)

transform = AddBackgroundNoise(
    sounds_path="ablation/ESC-50-master/audio",  # Replace with real path
    min_snr_db=5.0,
    max_snr_db=5.0,
    noise_transform=PolarityInversion(),
    p=1.0
)

with torch.inference_mode():
    encoder.eval()
    decoder.eval()
    discriminator.eval()
    test_avg_acc = [0, 0]
    test_avg_snr = 0
    count = 0
    for sample in track(dev_audios_loader):
        count += 1
        b = sample["matrix"].shape[0]
        # ---------------- build watermark
        wav_matrix = sample["matrix"].to(device)
        # Fixed message for testing (same for all in batch)
        msg = torch.tensor([-1.,  1.,  1., -1., -1.,  1., -1.,  1., -1., -1.], dtype=torch.float32)
        msg = msg.unsqueeze(0).repeat(b, 1, 1).to(device)

        # Watermark embedding
        watermark, carrier_wateramrked = encoder(wav_matrix, msg, 1)
        y_wm = wav_matrix + watermark

        # Convert each sample to NumPy, apply random noise, convert back
        # Convert each sample to NumPy, apply random noise, convert back
        augmented_waveforms = []
        for i in range(y_wm.shape[0]):
            waveform_np = y_wm[i].detach().cpu().numpy()
            if waveform_np.ndim > 1:
                waveform_np = waveform_np[0]  # Select first channel if stereo
            augmented_np = transform(samples=waveform_np, sample_rate=16000)
            augmented_waveforms.append(torch.tensor(augmented_np).unsqueeze(0))  # Add (1, L)

        y_wm_aug = torch.cat(augmented_waveforms, dim=0).to(device)  # Shape: (B, L)

        decoded = decoder(y_wm_aug, 1)
        decoder_acc = [((decoded[0] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item(),
                           ((decoded[1] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item()]
        zero_tensor = torch.zeros(wav_matrix.shape).to(device)
        snr = 10 * torch.log10(
            mse_loss(wav_matrix.detach(), zero_tensor) / mse_loss(wav_matrix.detach(), y_wm_aug.detach()))

        test_avg_snr += snr
        test_avg_acc[0] += decoder_acc[0]
        test_avg_acc[1] += decoder_acc[1]

    test_avg_acc[0] /= count
    test_avg_acc[1] /= count
    test_avg_snr /= count
    print("Test Average SNR:", test_avg_snr)
    print("Test Average Accuracy:", test_avg_acc[0])

Test Average SNR: tensor(4.9615, device='cuda:0')
Test Average Accuracy: 0.949999988079071


In [51]:
Audio(y_wm_aug[0].cpu(), rate=sr)

## Editing

### Normal Editing

#### LibriSpeech

In [52]:
irrelevant_paths = glob("/home/yizhu/Data/LJSpeech-1.1/test/*.wav")
train_config["path"]["raw_path"] = "/home/yizhu/Data/LibriSpeech_wav"
process_config["audio"]["or_sample_rate"]  = 16000
dev_audios = MyDataset(
    process_config=process_config, train_config=train_config, flag="test"
)
dev_audios_loader = DataLoader(
        dev_audios,
        batch_size=2,
        shuffle=False,
        collate_fn=collate_fn,
        pin_memory=True,
        num_workers=20,
        persistent_workers=True,
    )

In [53]:
with torch.inference_mode():
    encoder.eval()
    decoder.eval()
    discriminator.eval()
    test_avg_acc = [0, 0]
    test_avg_snr = 0
    count = 0
    for sample in track(dev_audios_loader):
        count += 1
        b = sample["matrix"].shape[0]
        # ---------------- build watermark
        wav_matrix = sample["matrix"].to(device)
        msg = generate_random_msg(wav_matrix.size(0), train_config["watermark"]["length"], device)
        watermark, carrier_wateramrked = encoder(wav_matrix, msg, 1)
        y_wm = wav_matrix + watermark

        y_wm_aug = splice_with_irrelevant(y_wm.unsqueeze(1), irrelevant_paths, sample_rate=16000)

        decoded = decoder(y_wm_aug.squeeze(1), 1)
        decoder_acc = [((decoded[0] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item(),
                           ((decoded[1] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item()]
        zero_tensor = torch.zeros(wav_matrix.shape).to(device)
        snr = 10 * torch.log10(
            mse_loss(wav_matrix.detach(), zero_tensor) / mse_loss(wav_matrix.detach(), y_wm_aug.detach()))

        test_avg_snr += snr
        test_avg_acc[0] += decoder_acc[0]
        test_avg_acc[1] += decoder_acc[1]

    test_avg_acc[0] /= count
    test_avg_acc[1] /= count
    test_avg_snr /= count
    print("Test Average SNR:", test_avg_snr)
    print("Test Average Accuracy:", test_avg_acc[0])

Test Average SNR: tensor(-5.3884, device='cuda:0')
Test Average Accuracy: 0.8611111177338494


In [54]:
Audio(y_wm_aug[0].cpu(), rate=sr)

#### LJSpeech

In [55]:
irrelevant_paths = glob("/home/yizhu/Data/LibriSpeech_wav/test/*.wav")
train_config["path"]["raw_path"] = "/home/yizhu/Data/LJSpeech-1.1"
process_config["audio"]["or_sample_rate"]  = 22050
dev_audios = MyDataset(
    process_config=process_config, train_config=train_config, flag="test"
)
dev_audios_loader = DataLoader(
        dev_audios,
        batch_size=2,
        shuffle=False,
        collate_fn=collate_fn,
        pin_memory=True,
        num_workers=20,
        persistent_workers=True,
    )

In [56]:
with torch.inference_mode():
    encoder.eval()
    decoder.eval()
    discriminator.eval()
    test_avg_acc = [0, 0]
    test_avg_snr = 0
    count = 0
    for sample in track(dev_audios_loader):
        count += 1
        b = sample["matrix"].shape[0]
        # ---------------- build watermark
        wav_matrix = sample["matrix"].to(device)
        msg = generate_random_msg(wav_matrix.size(0), train_config["watermark"]["length"], device)
        watermark, carrier_wateramrked = encoder(wav_matrix, msg, 1)
        y_wm = wav_matrix + watermark

        y_wm_aug = splice_with_irrelevant(y_wm.unsqueeze(1), irrelevant_paths, sample_rate=16000)

        decoded = decoder(y_wm_aug.squeeze(1), 1)
        decoder_acc = [((decoded[0] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item(),
                           ((decoded[1] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item()]
        zero_tensor = torch.zeros(wav_matrix.shape).to(device)
        snr = 10 * torch.log10(
            mse_loss(wav_matrix.detach(), zero_tensor) / mse_loss(wav_matrix.detach(), y_wm_aug.detach()))

        test_avg_snr += snr
        test_avg_acc[0] += decoder_acc[0]
        test_avg_acc[1] += decoder_acc[1]

    test_avg_acc[0] /= count
    test_avg_acc[1] /= count
    test_avg_snr /= count
    print("Test Average SNR:", test_avg_snr)
    print("Test Average Accuracy:", test_avg_acc[0])

Test Average SNR: tensor(-0.6063, device='cuda:0')
Test Average Accuracy: 0.7750000059604645


In [57]:
Audio(y_wm_aug[0].cpu(), rate=sr)

### Extreme Editing

#### LibriSpeech

In [58]:
irrelevant_paths = glob("/home/yizhu/Data/LJSpeech-1.1/test/*.wav")
train_config["path"]["raw_path"] = "/home/yizhu/Data/LibriSpeech_wav"
process_config["audio"]["or_sample_rate"]  = 16000
dev_audios = MyDataset(
    process_config=process_config, train_config=train_config, flag="test"
)
dev_audios_loader = DataLoader(
        dev_audios,
        batch_size=2,
        shuffle=False,
        collate_fn=collate_fn,
        pin_memory=True,
        num_workers=20,
        persistent_workers=True,
    )

In [59]:
with torch.inference_mode():
    encoder.eval()
    decoder.eval()
    discriminator.eval()
    test_avg_acc = [0, 0]
    test_avg_snr = 0
    count = 0
    for sample in track(dev_audios_loader):
        count += 1
        b = sample["matrix"].shape[0]
        # ---------------- build watermark
        wav_matrix = sample["matrix"].to(device)
        msg = generate_random_msg(wav_matrix.size(0), train_config["watermark"]["length"], device)
        watermark, carrier_wateramrked = encoder(wav_matrix, msg, 1)
        y_wm = wav_matrix + watermark

        y_wm_aug = splice_with_irrelevant_scattered(y_wm.unsqueeze(1), irrelevant_paths, sample_rate=16000, num_segments=5)

        decoded = decoder(y_wm_aug.squeeze(1), 1)
        decoder_acc = [((decoded[0] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item(),
                           ((decoded[1] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item()]
        zero_tensor = torch.zeros(wav_matrix.shape).to(device)
        snr = 10 * torch.log10(
            mse_loss(wav_matrix.detach(), zero_tensor) / mse_loss(wav_matrix.detach(), y_wm_aug.detach()))

        test_avg_snr += snr
        test_avg_acc[0] += decoder_acc[0]
        test_avg_acc[1] += decoder_acc[1]

    test_avg_acc[0] /= count
    test_avg_acc[1] /= count
    test_avg_snr /= count
    print("Test Average SNR:", test_avg_snr)
    print("Test Average Accuracy:", test_avg_acc[0])

Test Average SNR: tensor(-7.6692, device='cuda:0')
Test Average Accuracy: 0.800000011920929


In [60]:
Audio(y_wm[0].cpu(), rate=16000)

In [61]:
Audio(y_wm_aug[0].cpu(), rate=16000)

#### LJSpeech

In [62]:
irrelevant_paths = glob("/home/yizhu/Data/LibriSpeech_wav/test/*.wav")
train_config["path"]["raw_path"] = "/home/yizhu/Data/LJSpeech-1.1"
process_config["audio"]["or_sample_rate"]  = 22050
dev_audios = MyDataset(
    process_config=process_config, train_config=train_config, flag="test"
)
dev_audios_loader = DataLoader(
        dev_audios,
        batch_size=2,
        shuffle=False,
        collate_fn=collate_fn,
        pin_memory=True,
        num_workers=20,
        persistent_workers=True,
    )

In [63]:
with torch.inference_mode():
    encoder.eval()
    decoder.eval()
    discriminator.eval()
    test_avg_acc = [0, 0]
    test_avg_snr = 0
    count = 0
    for sample in track(dev_audios_loader):
        count += 1
        b = sample["matrix"].shape[0]
        # ---------------- build watermark
        wav_matrix = sample["matrix"].to(device)
        msg = generate_random_msg(wav_matrix.size(0), train_config["watermark"]["length"], device)
        watermark, carrier_wateramrked = encoder(wav_matrix, msg, 1)
        y_wm = wav_matrix + watermark

        y_wm_aug = splice_with_irrelevant_scattered(y_wm.unsqueeze(1), irrelevant_paths, sample_rate=16000, num_segments=5)

        decoded = decoder(y_wm_aug.squeeze(1), 1)
        decoder_acc = [((decoded[0] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item(),
                           ((decoded[1] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item()]
        zero_tensor = torch.zeros(wav_matrix.shape).to(device)
        snr = 10 * torch.log10(
            mse_loss(wav_matrix.detach(), zero_tensor) / mse_loss(wav_matrix.detach(), y_wm_aug.detach()))

        test_avg_snr += snr
        test_avg_acc[0] += decoder_acc[0]
        test_avg_acc[1] += decoder_acc[1]

    test_avg_acc[0] /= count
    test_avg_acc[1] /= count
    test_avg_snr /= count
    print("Test Average SNR:", test_avg_snr)
    print("Test Average Accuracy:", test_avg_acc[0])

Test Average SNR: tensor(-0.4878, device='cuda:0')
Test Average Accuracy: 0.7750000059604645


In [64]:
Audio(y_wm[0].cpu(), rate=16000)

In [65]:
Audio(y_wm_aug[0].cpu(), rate=16000)

# MSE + Loudness + Split Frequency

## Load Model Architecture

In [66]:
import torch.nn as nn
from torch.nn import LeakyReLU
from watermarking_model.model.blocks import FCBlock, Conv2Encoder, WatermarkEmbedder, WatermarkExtracter, ReluBlock
from distortions.frequency import TacotronSTFT, fixed_STFT, tacotron_mel
from silero_vad import load_silero_vad
import yaml
from dataset.data import WavDataset as MyDataset
from dataset.data import collate_fn
from torch.utils.data import DataLoader
import torch
import torchaudio
from rich.progress import track
from torch.nn.functional import mse_loss

# Optional: set up a small constant
EPS = 1e-9

class Encoder(nn.Module):
    def __init__(self, process_config, model_config, train_config, msg_length):
        super(Encoder, self).__init__()
        self.name = "conv2"
        self.win_dim = int((process_config["mel"]["n_fft"] / 2) + 1)
        self.add_carrier_noise = False
        self.block = model_config["conv2"]["block"]
        self.layers_CE = model_config["conv2"]["layers_CE"]
        self.EM_input_dim = model_config["conv2"]["hidden_dim"] + 3
        self.layers_EM = model_config["conv2"]["layers_EM"]
        self.n_fft = process_config["mel"]["n_fft"]
        self.hop_length = process_config["mel"]["hop_length"]
        self.win_length = process_config["mel"]["win_length"]
        self.sampling_rate = process_config["audio"]["or_sample_rate"]
        self.delay_amt = int((train_config["watermark"]["delay_amt_second"]*self.sampling_rate) // self.hop_length + 1)
        self.future_amt = int((train_config["watermark"]["future_amt_second"]*self.sampling_rate) // self.hop_length + 1)
        self.delay = True
        self.power = 1.0
        self.vad = load_silero_vad()
        self.vad_threshold = 0.50

        self.vocoder_step = model_config["structure"]["vocoder_step"]
        #MLP for the input wm
        self.msg_linear_in = FCBlock(msg_length, self.win_dim//2, activation=LeakyReLU(inplace=True))

        #stft transform
        self.stft = fixed_STFT(process_config["mel"]["n_fft"], process_config["mel"]["hop_length"], process_config["mel"]["win_length"])

        self.ENc = Conv2Encoder(input_channel=2, hidden_dim = model_config["conv2"]["hidden_dim"], block=self.block, n_layers=self.layers_CE)

        self.EM = WatermarkEmbedder(input_channel=self.EM_input_dim, hidden_dim = model_config["conv2"]["hidden_dim"], block=self.block, n_layers=self.layers_EM)

    def pad_w_zero_stft(self, input_stft, watermark_stft, voice_prefilling):
        """
        Pad the watermarked stft output with zeros on the left + right,
        respecting future_amt and chunk-based offsets.
        """
        chunk_size = voice_prefilling if not self.delay else (voice_prefilling + self.delay_amt)

        zeros_right_len = input_stft.shape[3] - watermark_stft.shape[3] - chunk_size
        if zeros_right_len < 0:
            # Edge case: won't happen if chunking logic is correct, but just to be safe
            zeros_right_len = 0

        zeros_left = torch.zeros_like(input_stft[:, :, :, :chunk_size])
        zeros_right = torch.zeros_like(input_stft[:, :, :, :zeros_right_len])

        actual_watermark = torch.cat([zeros_left, watermark_stft, zeros_right], dim=3) + EPS
        return actual_watermark

    def forward(self, x, msg, global_step):
        num_samples = x.shape[-1]
        _, _, stft_result = self.stft.transform(x)
        # Evaluate how many chunks we can process
        # 2s input + 0.5s calculation delay
        # 2.00*16000 = 32000
        # 32800 // hop_length + 1 = 201 center=True
        # 0.5s*16000 = 8000
        # 8000 // hop_length + 1 = 51 center=True
        voice_prefilling = int((2.00*self.sampling_rate)//self.hop_length + 1)
        # Predict future 0.5s watermark
        # 0.5*16000 = 8000
        # 8000 // hop_length + 1 =51
        max_start = stft_result.shape[-1] - (voice_prefilling + self.delay_amt)
        if int(max_start / self.delay_amt) <= 0:
            return None  # Not enough frames for a chunk

        list_of_watermarks = []
        for i in range(int((stft_result.shape[-1] - (voice_prefilling + self.delay_amt)) / self.future_amt)):
            carrier_encoded = self.ENc(stft_result[:, :, :, i * self.future_amt:voice_prefilling + i * self.future_amt])
            # torch.Size([B, 1, 81])
            # torch.Size([B, 81, 1])
            # torch.Size([B, 1, 81, 1])
            # torch.Size([B, 1, 162, 201])
            watermark_encoded = self.msg_linear_in(msg).transpose(1, 2).unsqueeze(1).repeat(1, 1, 2,
                                                                                            carrier_encoded.shape[3])
            concatenated_feature = torch.cat((carrier_encoded, stft_result[:, :, :,
                                                               i*self.future_amt:voice_prefilling + i*self.future_amt], watermark_encoded), dim=1)
            # [B, 2, bins, length]
            # Embed the watermark
            carrier_watermarked = self.EM(concatenated_feature)
            # Append both the watermark chunk and the pilot segment
            list_of_watermarks.append(carrier_watermarked)

        if len(list_of_watermarks) > 0:
            watermark = torch.cat(list_of_watermarks, dim=-1)
            all_watermark_stft = self.pad_w_zero_stft(
                stft_result, watermark, voice_prefilling
            )
            del list_of_watermarks
            mask=stft_result!=0
            all_watermark_stft = all_watermark_stft*mask + 0.0000001

            self.stft.num_samples = num_samples

            # Recompute magnitude & phase
            real_part = all_watermark_stft[:, 0, :, :]
            imag_part = all_watermark_stft[:, 1, :, :]
            spect = torch.sqrt(real_part ** 2 + imag_part ** 2)
            phase = torch.atan2(imag_part, real_part)

            y = self.stft.inverse(spect, phase).squeeze(1)
            del spect, phase, real_part, imag_part

            # with torch.no_grad():
            #     # Get chunk-level speech probabilities for the batch.
            #     # The output shape should be [batch, num_chunks]
            #     batch_chunk_probs = self.vad.audio_forward(x, sr=self.sampling_rate)
            #
            # # Threshold the probabilities to obtain a binary mask per chunk.
            # batch_chunk_mask = (batch_chunk_probs > self.vad_threshold).float()
            #
            # # Upsample the chunk-level mask to a sample-level mask.
            # # Each chunk's decision is repeated for chunk_size samples.
            # sample_masks = torch.repeat_interleave(batch_chunk_mask, 512, dim=1).to(y.device)
            #
            # # Since the upsampled mask might be longer than the actual audio length,
            # # slice the mask to match the original number of samples.
            # sample_length = x.shape[-1]
            # sample_masks = sample_masks[:, :sample_length]
            #
            # # Apply the mask to the original audio to zero out non-speech regions.
            # masked_y = y * sample_masks
            return y, all_watermark_stft
        else:
            print("Not enough watermarking!!!!")
            return None


class Decoder(nn.Module):
    def __init__(self, process_config, model_config, train_config, msg_length):
        super(Decoder, self).__init__()
        self.robust = model_config["robust"]
        # if self.robust:
        #     self.dl = distortion(process_config, train_config)
        self.mel_transform = TacotronSTFT(filter_length=process_config["mel"]["n_fft"], hop_length=process_config["mel"]["hop_length"], win_length=process_config["mel"]["win_length"])
        # self.vocoder = get_vocoder(device)
        self.vocoder_step = model_config["structure"]["vocoder_step"]
        self.win_dim = int((process_config["mel"]["n_fft"] / 2) + 1)
        self.hop_length = process_config["mel"]["hop_length"]
        self.block = model_config["conv2"]["block"]
        self.EX = WatermarkExtracter(input_channel=2, hidden_dim=model_config["conv2"]["hidden_dim"], block=self.block)
        self.stft = fixed_STFT(process_config["mel"]["n_fft"], process_config["mel"]["hop_length"], process_config["mel"]["win_length"])
        self.msg_linear_out = FCBlock(self.win_dim//2, msg_length, activation=LeakyReLU(inplace=True))

    def forward(self, y, global_step):
        y_identity = y
        # if global_step > self.vocoder_step:
        #     y_mel = self.mel_transform.mel_spectrogram(y.squeeze(1))
        #     # y = self.vocoder(y_mel)
        #     y_d = (self.mel_transform.griffin_lim(magnitudes=y_mel)).unsqueeze(1)
        # else:
        #     y_d = y
        y_d = y

        spect, phase, stft_result = self.stft.transform(y_d.squeeze(1))
        extracted_wm = self.EX(stft_result).squeeze(1)  # (B, win_dim, length)
        # Explicitly split the 162-dim vector into two halves of 81-dim each
        low, high = extracted_wm.chunk(2, dim=1)  # each has shape [B, win_dim / 2, length]
        low_msg = torch.mean(low, dim=2, keepdim=True).transpose(1,2)
        high_msg = torch.mean(high, dim=2, keepdim=True).transpose(1, 2)
        msg_avg = (low_msg + high_msg) / 2  # Average the two halves -> shape: [B, 1, 81]
        # msg = torch.mean(extracted_wm, dim=2, keepdim=True).transpose(1,2)
        # msg = self.msg_linear_out(msg)
        msg = self.msg_linear_out(msg_avg)

        _, _, stft_result_identity = self.stft.transform(y_identity)
        extracted_wm_identity = self.EX(stft_result_identity).squeeze(1)
        low_identity, high_identity = extracted_wm_identity.chunk(2, dim=1)  # each has shape [B, win_dim / 2, length]
        low_msg_identity = torch.mean(low_identity, dim=2, keepdim=True).transpose(1, 2)
        high_msg_identity = torch.mean(high_identity, dim=2, keepdim=True).transpose(1, 2)
        msg_avg_identity = (low_msg_identity + high_msg_identity) / 2  # Average the two halves -> shape: [B, 1, 81]
        # msg_identity = torch.mean(extracted_wm_identity,dim=2, keepdim=True).transpose(1,2)
        # msg_identity = self.msg_linear_out(msg_identity)
        msg_identity = self.msg_linear_out(msg_avg_identity)
        del stft_result, stft_result_identity, extracted_wm, extracted_wm_identity
        return msg, msg_identity

class Discriminator(nn.Module):
    def __init__(self, process_config):
        super(Discriminator, self).__init__()
        self.conv = nn.Sequential(
                ReluBlock(2,16,3,1,1),
                ReluBlock(16,32,3,1,1),
                ReluBlock(32,64,3,1,1),
                nn.AdaptiveAvgPool2d(output_size=(1, 1))
                )
        self.linear = nn.Linear(64,1)
        self.stft = fixed_STFT(process_config["mel"]["n_fft"], process_config["mel"]["hop_length"], process_config["mel"]["win_length"])

    def forward(self, x):
        _, _, stft_result = self.stft.transform(x)
        x = self.conv(stft_result)
        x = x.squeeze(2).squeeze(2)
        x = self.linear(x)
        return x

In [67]:
process_config = yaml.load(open("./config/process.yaml", "r"), Loader=yaml.FullLoader)
model_config = yaml.load(open("./config/model.yaml", "r"), Loader=yaml.FullLoader)
train_config = yaml.load(open("./config/train.yaml", "r"), Loader=yaml.FullLoader)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

encoder = Encoder(process_config, model_config, train_config, train_config["watermark"]["length"]).to(device)
decoder = Decoder(process_config, model_config, train_config, train_config["watermark"]["length"]).to(device)
discriminator = Discriminator(process_config).to(device)
checkpoint_path = "ablation/original_split_frequency_conv2_ep_20_2025-04-28_15_59_35.pth.tar"
checkpoint = torch.load(checkpoint_path, map_location=device)
encoder.load_state_dict(checkpoint["encoder"])
decoder.load_state_dict(checkpoint["decoder"])

<All keys matched successfully>

## Single Inference

In [32]:
with torch.inference_mode():
    encoder.eval()
    decoder.eval()
    msg = generate_random_msg(1, train_config["watermark"]["length"], device)
    # wav, sr = torchaudio.load("ablation/260-123440-0016.wav")
    # wav, sr = torchaudio.load("ablation/LJ001-0052.wav")
    wav, sr = torchaudio.load("ablation/LJ001-0053.wav")
    wav = wav.to(device) # (batch, length)
    watermark, carrier_wateramrked = encoder(wav, msg, 1) # (batch, 1, length)
    y_wm = wav + watermark
    decoded = decoder(y_wm, 1)
    decoder_acc = [((decoded[0] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item(),
                           ((decoded[1] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item()]
    print("Accuracy:[{:.8f},{:.8f}]".format(*decoder_acc))
    print("original msg:", msg)
    print("decoded msg:", decoded[0])

    # y_norm = normalize_audio(wav.cpu().detach())
    wm_norm = normalize_audio(watermark[0].cpu().detach())
    y_wm_norm = normalize_audio(y_wm[0].cpu().detach())

    # save_spectrogram(y_norm.squeeze(0), "ablation/orig_spectrogram.png")
    save_spectrogram(wm_norm, "ablation/MSE_loudness_split_frequency_watermark_spectrogram.png")
    save_spectrogram(y_wm_norm, "ablation/MSE_loudness_split_frequency_watermarked_spectrogram.png")

    # save_audio(wav, "ablation/original_audio.wav", sample_rate=sr)
    save_audio(watermark, "ablation/MSE_loudness_split_frequency_watermark_audio.wav", sample_rate=sr)
    save_audio(y_wm, "ablation/MSE_loudness_split_frequency_watermarked_audio.wav", sample_rate=sr)

Accuracy:[1.00000000,1.00000000]
original msg: tensor([[[-1., -1.,  1.,  1.,  1., -1., -1., -1., -1., -1.]]], device='cuda:0')
decoded msg: tensor([[[-0.0130, -0.0126,  0.8006,  0.8165,  0.7504, -0.0134, -0.0092,
          -0.0124, -0.0129, -0.0080]]], device='cuda:0')


### Original Audio

In [17]:
Audio(wav.cpu(), rate=sr)

### Watermarked Audio

In [18]:
Audio(y_wm.cpu(), rate=sr)

## Batch Inference

### LibriSpeech

In [ ]:
train_config["path"]["raw_path"] = "/home/yizhu/Data/LibriSpeech_wav"
dev_audios = MyDataset(
    process_config=process_config, train_config=train_config, flag="test"
)
dev_audios_loader = DataLoader(
        dev_audios,
        batch_size=2,
        shuffle=False,
        collate_fn=collate_fn,
        pin_memory=True,
        num_workers=20,
        persistent_workers=True,
    )

In [ ]:
with torch.inference_mode():
    encoder.eval()
    decoder.eval()
    discriminator.eval()
    test_avg_acc = [0, 0]
    test_avg_snr = 0
    count = 0
    for sample in track(dev_audios_loader):
        count += 1
        b = sample["matrix"].shape[0]
        # ---------------- build watermark
        wav_matrix = sample["matrix"].to(device)
        msg = generate_random_msg(wav_matrix.size(0), train_config["watermark"]["length"], device)
        watermark, carrier_wateramrked = encoder(wav_matrix, msg, 1)
        y_wm = wav_matrix + watermark
        decoded = decoder(y_wm, 1)
        decoder_acc = [((decoded[0] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item(),
                           ((decoded[1] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item()]
        zero_tensor = torch.zeros(wav_matrix.shape).to(device)
        snr = 10 * torch.log10(
            mse_loss(wav_matrix.detach(), zero_tensor) / mse_loss(wav_matrix.detach(), y_wm.detach()))

        test_avg_snr += snr
        test_avg_acc[0] += decoder_acc[0]
        test_avg_acc[1] += decoder_acc[1]

    test_avg_acc[0] /= count
    test_avg_acc[1] /= count
    test_avg_snr /= count
    print("Test Average SNR:", test_avg_snr)
    print("Test Average Accuracy:", test_avg_acc[0])

In [ ]:
count

### LJSpeech

In [ ]:
train_config["path"]["raw_path"] = "/home/yizhu/Data/LJSpeech-1.1"
dev_audios = MyDataset(
    process_config=process_config, train_config=train_config, flag="test"
)
dev_audios_loader = DataLoader(
        dev_audios,
        batch_size=2,
        shuffle=False,
        collate_fn=collate_fn,
        pin_memory=True,
        num_workers=20,
        persistent_workers=True,
    )

In [ ]:
with torch.inference_mode():
    encoder.eval()
    decoder.eval()
    discriminator.eval()
    test_avg_acc = [0, 0]
    test_avg_snr = 0
    count = 0
    for sample in track(dev_audios_loader):
        count += 1
        b = sample["matrix"].shape[0]
        # ---------------- build watermark
        wav_matrix = sample["matrix"].to(device)
        msg = generate_random_msg(wav_matrix.size(0), train_config["watermark"]["length"], device)
        watermark, carrier_wateramrked = encoder(wav_matrix, msg, 1)
        y_wm = wav_matrix + watermark
        decoded = decoder(y_wm, 1)
        decoder_acc = [((decoded[0] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item(),
                           ((decoded[1] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item()]
        zero_tensor = torch.zeros(wav_matrix.shape).to(device)
        snr = 10 * torch.log10(
            mse_loss(wav_matrix.detach(), zero_tensor) / mse_loss(wav_matrix.detach(), y_wm_aug.detach()))

        test_avg_snr += snr
        test_avg_acc[0] += decoder_acc[0]
        test_avg_acc[1] += decoder_acc[1]

    test_avg_acc[0] /= count
    test_avg_acc[1] /= count
    test_avg_snr /= count
    print("Test Average SNR:", test_avg_snr)
    print("Test Average Accuracy:", test_avg_acc[0])

## Background Noise

### 3-30 snr db

#### LibriSpeech

In [ ]:
train_config["path"]["raw_path"] = "/home/yizhu/Data/LibriSpeech_wav"
dev_audios = MyDataset(
    process_config=process_config, train_config=train_config, flag="test"
)
dev_audios_loader = DataLoader(
        dev_audios,
        batch_size=2,
        shuffle=False,
        collate_fn=collate_fn,
        pin_memory=True,
        num_workers=20,
        persistent_workers=True,
    )

In [ ]:
from audiomentations import AddBackgroundNoise, PolarityInversion

# Fix randomness
random.seed(42)
np.random.seed(42)

transform = AddBackgroundNoise(
    sounds_path="ablation/ESC-50-master/audio",  # Replace with real path
    min_snr_db=3.0,
    max_snr_db=30.0,
    noise_transform=PolarityInversion(),
    p=1.0
)

with torch.inference_mode():
    encoder.eval()
    decoder.eval()
    discriminator.eval()
    test_avg_acc = [0, 0]
    test_avg_snr = 0
    count = 0
    for sample in track(dev_audios_loader):
        count += 1
        b = sample["matrix"].shape[0]
        # ---------------- build watermark
        wav_matrix = sample["matrix"].to(device)
        # Fixed message for testing (same for all in batch)
        msg = torch.tensor([-1.,  1.,  1., -1., -1.,  1., -1.,  1., -1., -1.], dtype=torch.float32)
        msg = msg.unsqueeze(0).repeat(b, 1, 1).to(device)

        # Watermark embedding
        watermark, carrier_wateramrked = encoder(wav_matrix, msg, 1)
        y_wm = wav_matrix + watermark

        # Convert each sample to NumPy, apply random noise, convert back
        # Convert each sample to NumPy, apply random noise, convert back
        augmented_waveforms = []
        for i in range(y_wm.shape[0]):
            waveform_np = y_wm[i].detach().cpu().numpy()
            if waveform_np.ndim > 1:
                waveform_np = waveform_np[0]  # Select first channel if stereo
            augmented_np = transform(samples=waveform_np, sample_rate=16000)
            augmented_waveforms.append(torch.tensor(augmented_np).unsqueeze(0))  # Add (1, L)

        y_wm_aug = torch.cat(augmented_waveforms, dim=0).to(device)  # Shape: (B, L)

        decoded = decoder(y_wm_aug, 1)
        decoder_acc = [((decoded[0] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item(),
                           ((decoded[1] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item()]
        zero_tensor = torch.zeros(wav_matrix.shape).to(device)
        snr = 10 * torch.log10(
            mse_loss(wav_matrix.detach(), zero_tensor) / mse_loss(wav_matrix.detach(), y_wm_aug.detach()))

        test_avg_snr += snr
        test_avg_acc[0] += decoder_acc[0]
        test_avg_acc[1] += decoder_acc[1]

    test_avg_acc[0] /= count
    test_avg_acc[1] /= count
    test_avg_snr /= count
    print("Test Average SNR:", test_avg_snr)
    print("Test Average Accuracy:", test_avg_acc[0])

#### LJSpeech

In [ ]:
train_config["path"]["raw_path"] = "/home/yizhu/Data/LJSpeech-1.1"
dev_audios = MyDataset(
    process_config=process_config, train_config=train_config, flag="test"
)
dev_audios_loader = DataLoader(
        dev_audios,
        batch_size=2,
        shuffle=False,
        collate_fn=collate_fn,
        pin_memory=True,
        num_workers=20,
        persistent_workers=True,
    )

In [ ]:
from audiomentations import AddBackgroundNoise, PolarityInversion

# Fix randomness
random.seed(42)
np.random.seed(42)

transform = AddBackgroundNoise(
    sounds_path="ablation/ESC-50-master/audio",  # Replace with real path
    min_snr_db=3.0,
    max_snr_db=30.0,
    noise_transform=PolarityInversion(),
    p=1.0
)

with torch.inference_mode():
    encoder.eval()
    decoder.eval()
    discriminator.eval()
    test_avg_acc = [0, 0]
    test_avg_snr = 0
    count = 0
    for sample in track(dev_audios_loader):
        count += 1
        b = sample["matrix"].shape[0]
        # ---------------- build watermark
        wav_matrix = sample["matrix"].to(device)
        # Fixed message for testing (same for all in batch)
        msg = torch.tensor([-1.,  1.,  1., -1., -1.,  1., -1.,  1., -1., -1.], dtype=torch.float32)
        msg = msg.unsqueeze(0).repeat(b, 1, 1).to(device)

        # Watermark embedding
        watermark, carrier_wateramrked = encoder(wav_matrix, msg, 1)
        y_wm = wav_matrix + watermark

        # Convert each sample to NumPy, apply random noise, convert back
        # Convert each sample to NumPy, apply random noise, convert back
        augmented_waveforms = []
        for i in range(y_wm.shape[0]):
            waveform_np = y_wm[i].detach().cpu().numpy()
            if waveform_np.ndim > 1:
                waveform_np = waveform_np[0]  # Select first channel if stereo
            augmented_np = transform(samples=waveform_np, sample_rate=16000)
            augmented_waveforms.append(torch.tensor(augmented_np).unsqueeze(0))  # Add (1, L)

        y_wm_aug = torch.cat(augmented_waveforms, dim=0).to(device)  # Shape: (B, L)

        decoded = decoder(y_wm_aug, 1)
        decoder_acc = [((decoded[0] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item(),
                           ((decoded[1] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item()]
        zero_tensor = torch.zeros(wav_matrix.shape).to(device)
        snr = 10 * torch.log10(
            mse_loss(wav_matrix.detach(), zero_tensor) / mse_loss(wav_matrix.detach(), y_wm_aug.detach()))

        test_avg_snr += snr
        test_avg_acc[0] += decoder_acc[0]
        test_avg_acc[1] += decoder_acc[1]

    test_avg_acc[0] /= count
    test_avg_acc[1] /= count
    test_avg_snr /= count
    print("Test Average SNR:", test_avg_snr)
    print("Test Average Accuracy:", test_avg_acc[0])

### 5 snr db

#### LibriSpeech

In [ ]:
train_config["path"]["raw_path"] = "/home/yizhu/Data/LibriSpeech_wav"
dev_audios = MyDataset(
    process_config=process_config, train_config=train_config, flag="test"
)
dev_audios_loader = DataLoader(
        dev_audios,
        batch_size=2,
        shuffle=False,
        collate_fn=collate_fn,
        pin_memory=True,
        num_workers=20,
        persistent_workers=True,
    )

In [ ]:
from audiomentations import AddBackgroundNoise, PolarityInversion

# Fix randomness
random.seed(42)
np.random.seed(42)

transform = AddBackgroundNoise(
    sounds_path="ablation/ESC-50-master/audio",  # Replace with real path
    min_snr_db=5.0,
    max_snr_db=5.0,
    noise_transform=PolarityInversion(),
    p=1.0
)

with torch.inference_mode():
    encoder.eval()
    decoder.eval()
    discriminator.eval()
    test_avg_acc = [0, 0]
    test_avg_snr = 0
    count = 0
    for sample in track(dev_audios_loader):
        count += 1
        b = sample["matrix"].shape[0]
        # ---------------- build watermark
        wav_matrix = sample["matrix"].to(device)
        # Fixed message for testing (same for all in batch)
        msg = torch.tensor([-1.,  1.,  1., -1., -1.,  1., -1.,  1., -1., -1.], dtype=torch.float32)
        msg = msg.unsqueeze(0).repeat(b, 1, 1).to(device)

        # Watermark embedding
        watermark, carrier_wateramrked = encoder(wav_matrix, msg, 1)
        y_wm = wav_matrix + watermark

        # Convert each sample to NumPy, apply random noise, convert back
        # Convert each sample to NumPy, apply random noise, convert back
        augmented_waveforms = []
        for i in range(y_wm.shape[0]):
            waveform_np = y_wm[i].detach().cpu().numpy()
            if waveform_np.ndim > 1:
                waveform_np = waveform_np[0]  # Select first channel if stereo
            augmented_np = transform(samples=waveform_np, sample_rate=16000)
            augmented_waveforms.append(torch.tensor(augmented_np).unsqueeze(0))  # Add (1, L)

        y_wm_aug = torch.cat(augmented_waveforms, dim=0).to(device)  # Shape: (B, L)

        decoded = decoder(y_wm_aug, 1)
        decoder_acc = [((decoded[0] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item(),
                           ((decoded[1] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item()]
        zero_tensor = torch.zeros(wav_matrix.shape).to(device)
        snr = 10 * torch.log10(
            mse_loss(wav_matrix.detach(), zero_tensor) / mse_loss(wav_matrix.detach(), y_wm_aug.detach()))

        test_avg_snr += snr
        test_avg_acc[0] += decoder_acc[0]
        test_avg_acc[1] += decoder_acc[1]

    test_avg_acc[0] /= count
    test_avg_acc[1] /= count
    test_avg_snr /= count
    print("Test Average SNR:", test_avg_snr)
    print("Test Average Accuracy:", test_avg_acc[0])

#### LJSpeech

In [ ]:
train_config["path"]["raw_path"] = "/home/yizhu/Data/LJSpeech-1.1"
dev_audios = MyDataset(
    process_config=process_config, train_config=train_config, flag="test"
)
dev_audios_loader = DataLoader(
        dev_audios,
        batch_size=2,
        shuffle=False,
        collate_fn=collate_fn,
        pin_memory=True,
        num_workers=20,
        persistent_workers=True,
    )

In [ ]:
from audiomentations import AddBackgroundNoise, PolarityInversion

# Fix randomness
random.seed(42)
np.random.seed(42)

transform = AddBackgroundNoise(
    sounds_path="ablation/ESC-50-master/audio",  # Replace with real path
    min_snr_db=5.0,
    max_snr_db=5.0,
    noise_transform=PolarityInversion(),
    p=1.0
)

with torch.inference_mode():
    encoder.eval()
    decoder.eval()
    discriminator.eval()
    test_avg_acc = [0, 0]
    test_avg_snr = 0
    count = 0
    for sample in track(dev_audios_loader):
        count += 1
        b = sample["matrix"].shape[0]
        # ---------------- build watermark
        wav_matrix = sample["matrix"].to(device)
        # Fixed message for testing (same for all in batch)
        msg = torch.tensor([-1.,  1.,  1., -1., -1.,  1., -1.,  1., -1., -1.], dtype=torch.float32)
        msg = msg.unsqueeze(0).repeat(b, 1, 1).to(device)

        # Watermark embedding
        watermark, carrier_wateramrked = encoder(wav_matrix, msg, 1)
        y_wm = wav_matrix + watermark

        # Convert each sample to NumPy, apply random noise, convert back
        # Convert each sample to NumPy, apply random noise, convert back
        augmented_waveforms = []
        for i in range(y_wm.shape[0]):
            waveform_np = y_wm[i].detach().cpu().numpy()
            if waveform_np.ndim > 1:
                waveform_np = waveform_np[0]  # Select first channel if stereo
            augmented_np = transform(samples=waveform_np, sample_rate=16000)
            augmented_waveforms.append(torch.tensor(augmented_np).unsqueeze(0))  # Add (1, L)

        y_wm_aug = torch.cat(augmented_waveforms, dim=0).to(device)  # Shape: (B, L)

        decoded = decoder(y_wm_aug, 1)
        decoder_acc = [((decoded[0] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item(),
                           ((decoded[1] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item()]
        zero_tensor = torch.zeros(wav_matrix.shape).to(device)
        snr = 10 * torch.log10(
            mse_loss(wav_matrix.detach(), zero_tensor) / mse_loss(wav_matrix.detach(), y_wm_aug.detach()))

        test_avg_snr += snr
        test_avg_acc[0] += decoder_acc[0]
        test_avg_acc[1] += decoder_acc[1]

    test_avg_acc[0] /= count
    test_avg_acc[1] /= count
    test_avg_snr /= count
    print("Test Average SNR:", test_avg_snr)
    print("Test Average Accuracy:", test_avg_acc[0])

## Editing

### Normal Editing

#### LibriSpeech

In [ ]:
irrelevant_paths = glob("/home/yizhu/Data/LJSpeech-1.1/test/*.wav")
train_config["path"]["raw_path"] = "/home/yizhu/Data/LibriSpeech_wav"
dev_audios = MyDataset(
    process_config=process_config, train_config=train_config, flag="test"
)
dev_audios_loader = DataLoader(
        dev_audios,
        batch_size=2,
        shuffle=False,
        collate_fn=collate_fn,
        pin_memory=True,
        num_workers=20,
        persistent_workers=True,
    )

In [ ]:
with torch.inference_mode():
    encoder.eval()
    decoder.eval()
    discriminator.eval()
    test_avg_acc = [0, 0]
    test_avg_snr = 0
    count = 0
    for sample in track(dev_audios_loader):
        count += 1
        b = sample["matrix"].shape[0]
        # ---------------- build watermark
        wav_matrix = sample["matrix"].to(device)
        msg = generate_random_msg(wav_matrix.size(0), train_config["watermark"]["length"], device)
        watermark, carrier_wateramrked = encoder(wav_matrix, msg, 1)
        y_wm = wav_matrix + watermark

        y_wm_aug = splice_with_irrelevant(y_wm.unsqueeze(1), irrelevant_paths, sample_rate=16000)

        decoded = decoder(y_wm_aug.squeeze(1), 1)
        decoder_acc = [((decoded[0] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item(),
                           ((decoded[1] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item()]
        zero_tensor = torch.zeros(wav_matrix.shape).to(device)
        snr = 10 * torch.log10(
            mse_loss(wav_matrix.detach(), zero_tensor) / mse_loss(wav_matrix.detach(), y_wm_aug.detach()))

        test_avg_snr += snr
        test_avg_acc[0] += decoder_acc[0]
        test_avg_acc[1] += decoder_acc[1]

    test_avg_acc[0] /= count
    test_avg_acc[1] /= count
    test_avg_snr /= count
    print("Test Average SNR:", test_avg_snr)
    print("Test Average Accuracy:", test_avg_acc[0])

#### LJSpeech

In [ ]:
irrelevant_paths = glob("/home/yizhu/Data/LibriSpeech_wav/test_orig/*.wav")
train_config["path"]["raw_path"] = "/home/yizhu/Data/LJSpeech-1.1"
dev_audios = MyDataset(
    process_config=process_config, train_config=train_config, flag="test"
)
dev_audios_loader = DataLoader(
        dev_audios,
        batch_size=2,
        shuffle=False,
        collate_fn=collate_fn,
        pin_memory=True,
        num_workers=20,
        persistent_workers=True,
    )

In [ ]:
with torch.inference_mode():
    encoder.eval()
    decoder.eval()
    discriminator.eval()
    test_avg_acc = [0, 0]
    test_avg_snr = 0
    count = 0
    for sample in track(dev_audios_loader):
        count += 1
        b = sample["matrix"].shape[0]
        # ---------------- build watermark
        wav_matrix = sample["matrix"].to(device)
        msg = generate_random_msg(wav_matrix.size(0), train_config["watermark"]["length"], device)
        watermark, carrier_wateramrked = encoder(wav_matrix, msg, 1)
        y_wm = wav_matrix + watermark

        y_wm_aug = splice_with_irrelevant(y_wm.unsqueeze(1), irrelevant_paths, sample_rate=16000)

        decoded = decoder(y_wm_aug.squeeze(1), 1)
        decoder_acc = [((decoded[0] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item(),
                           ((decoded[1] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item()]
        zero_tensor = torch.zeros(wav_matrix.shape).to(device)
        snr = 10 * torch.log10(
            mse_loss(wav_matrix.detach(), zero_tensor) / mse_loss(wav_matrix.detach(), y_wm_aug.detach()))

        test_avg_snr += snr
        test_avg_acc[0] += decoder_acc[0]
        test_avg_acc[1] += decoder_acc[1]

    test_avg_acc[0] /= count
    test_avg_acc[1] /= count
    test_avg_snr /= count
    print("Test Average SNR:", test_avg_snr)
    print("Test Average Accuracy:", test_avg_acc[0])

### Extreme Editing

#### LibriSpeech

In [17]:
irrelevant_paths = glob("/home/yizhu/Data/LJSpeech-1.1/test/*.wav")
train_config["path"]["raw_path"] = "/home/yizhu/Data/LibriSpeech_wav"
process_config["audio"]["or_sample_rate"]  = 16000
dev_audios = MyDataset(
    process_config=process_config, train_config=train_config, flag="test"
)
dev_audios_loader = DataLoader(
        dev_audios,
        batch_size=2,
        shuffle=False,
        collate_fn=collate_fn,
        pin_memory=True,
        num_workers=20,
        persistent_workers=True,
    )

In [18]:
with torch.inference_mode():
    encoder.eval()
    decoder.eval()
    discriminator.eval()
    test_avg_acc = [0, 0]
    test_avg_snr = 0
    count = 0
    for sample in track(dev_audios_loader):
        count += 1
        b = sample["matrix"].shape[0]
        # ---------------- build watermark
        wav_matrix = sample["matrix"].to(device)
        msg = generate_random_msg(wav_matrix.size(0), train_config["watermark"]["length"], device)
        watermark, carrier_wateramrked = encoder(wav_matrix, msg, 1)
        y_wm = wav_matrix + watermark

        y_wm_aug = splice_with_irrelevant_scattered(y_wm.unsqueeze(1), irrelevant_paths, sample_rate=16000, num_segments=5)

        decoded = decoder(y_wm_aug.squeeze(1), 1)
        decoder_acc = [((decoded[0] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item(),
                           ((decoded[1] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item()]
        zero_tensor = torch.zeros(wav_matrix.shape).to(device)
        snr = 10 * torch.log10(
            mse_loss(wav_matrix.detach(), zero_tensor) / mse_loss(wav_matrix.detach(), y_wm_aug.detach()))

        test_avg_snr += snr
        test_avg_acc[0] += decoder_acc[0]
        test_avg_acc[1] += decoder_acc[1]

    test_avg_acc[0] /= count
    test_avg_acc[1] /= count
    test_avg_snr /= count
    print("Test Average SNR:", test_avg_snr)
    print("Test Average Accuracy:", test_avg_acc[0])

Test Average SNR: tensor(-3.5563, device='cuda:0')
Test Average Accuracy: 0.8993722044299002


#### LJSpeech

In [19]:
irrelevant_paths = glob("/home/yizhu/Data/LibriSpeech_wav/test/*.wav")
train_config["path"]["raw_path"] = "/home/yizhu/Data/LJSpeech-1.1"
process_config["audio"]["or_sample_rate"]  = 22050
dev_audios = MyDataset(
    process_config=process_config, train_config=train_config, flag="test"
)
dev_audios_loader = DataLoader(
        dev_audios,
        batch_size=2,
        shuffle=False,
        collate_fn=collate_fn,
        pin_memory=True,
        num_workers=20,
        persistent_workers=True,
    )

In [20]:
with torch.inference_mode():
    encoder.eval()
    decoder.eval()
    discriminator.eval()
    test_avg_acc = [0, 0]
    test_avg_snr = 0
    count = 0
    for sample in track(dev_audios_loader):
        count += 1
        b = sample["matrix"].shape[0]
        # ---------------- build watermark
        wav_matrix = sample["matrix"].to(device)
        msg = generate_random_msg(wav_matrix.size(0), train_config["watermark"]["length"], device)
        watermark, carrier_wateramrked = encoder(wav_matrix, msg, 1)
        y_wm = wav_matrix + watermark

        y_wm_aug = splice_with_irrelevant_scattered(y_wm.unsqueeze(1), irrelevant_paths, sample_rate=16000, num_segments=5)

        decoded = decoder(y_wm_aug.squeeze(1), 1)
        decoder_acc = [((decoded[0] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item(),
                           ((decoded[1] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item()]
        zero_tensor = torch.zeros(wav_matrix.shape).to(device)
        snr = 10 * torch.log10(
            mse_loss(wav_matrix.detach(), zero_tensor) / mse_loss(wav_matrix.detach(), y_wm_aug.detach()))

        test_avg_snr += snr
        test_avg_acc[0] += decoder_acc[0]
        test_avg_acc[1] += decoder_acc[1]

    test_avg_acc[0] /= count
    test_avg_acc[1] /= count
    test_avg_snr /= count
    print("Test Average SNR:", test_avg_snr)
    print("Test Average Accuracy:", test_avg_acc[0])

Test Average SNR: tensor(-1.9868, device='cuda:0')
Test Average Accuracy: 0.9291516292191037


In [21]:
Audio(y_wm[0].cpu(), rate=16000)

In [22]:
Audio(y_wm_aug[0].cpu(), rate=16000)

# MSE + Loudness + Split Frequency + Vad

## Load Model Architecture

In [27]:
import torch.nn as nn
from torch.nn import LeakyReLU
from watermarking_model.model.blocks import FCBlock, Conv2Encoder, WatermarkEmbedder, WatermarkExtracter, ReluBlock
from distortions.frequency import TacotronSTFT, fixed_STFT, tacotron_mel
from silero_vad import load_silero_vad
import yaml
from dataset.data import WavDataset as MyDataset
from dataset.data import collate_fn
from torch.utils.data import DataLoader
import torch
import torchaudio
from rich.progress import track
from torch.nn.functional import mse_loss

# Optional: set up a small constant
EPS = 1e-9

class Encoder(nn.Module):
    def __init__(self, process_config, model_config, train_config, msg_length):
        super(Encoder, self).__init__()
        self.name = "conv2"
        self.win_dim = int((process_config["mel"]["n_fft"] / 2) + 1)
        self.add_carrier_noise = False
        self.block = model_config["conv2"]["block"]
        self.layers_CE = model_config["conv2"]["layers_CE"]
        self.EM_input_dim = model_config["conv2"]["hidden_dim"] + 3
        self.layers_EM = model_config["conv2"]["layers_EM"]
        self.n_fft = process_config["mel"]["n_fft"]
        self.hop_length = process_config["mel"]["hop_length"]
        self.win_length = process_config["mel"]["win_length"]
        self.sampling_rate = process_config["audio"]["or_sample_rate"]
        self.delay_amt = int((train_config["watermark"]["delay_amt_second"]*self.sampling_rate) // self.hop_length + 1)
        self.future_amt = int((train_config["watermark"]["future_amt_second"]*self.sampling_rate) // self.hop_length + 1)
        self.delay = True
        self.power = 1.0
        self.vad = load_silero_vad()
        self.vad_threshold = 0.50

        self.vocoder_step = model_config["structure"]["vocoder_step"]
        #MLP for the input wm
        self.msg_linear_in = FCBlock(msg_length, self.win_dim//2, activation=LeakyReLU(inplace=True))

        #stft transform
        self.stft = fixed_STFT(process_config["mel"]["n_fft"], process_config["mel"]["hop_length"], process_config["mel"]["win_length"])

        self.ENc = Conv2Encoder(input_channel=2, hidden_dim = model_config["conv2"]["hidden_dim"], block=self.block, n_layers=self.layers_CE)

        self.EM = WatermarkEmbedder(input_channel=self.EM_input_dim, hidden_dim = model_config["conv2"]["hidden_dim"], block=self.block, n_layers=self.layers_EM)

    def pad_w_zero_stft(self, input_stft, watermark_stft, voice_prefilling):
        """
        Pad the watermarked stft output with zeros on the left + right,
        respecting future_amt and chunk-based offsets.
        """
        chunk_size = voice_prefilling if not self.delay else (voice_prefilling + self.delay_amt)

        zeros_right_len = input_stft.shape[3] - watermark_stft.shape[3] - chunk_size
        if zeros_right_len < 0:
            # Edge case: won't happen if chunking logic is correct, but just to be safe
            zeros_right_len = 0

        zeros_left = torch.zeros_like(input_stft[:, :, :, :chunk_size])
        zeros_right = torch.zeros_like(input_stft[:, :, :, :zeros_right_len])

        actual_watermark = torch.cat([zeros_left, watermark_stft, zeros_right], dim=3) + EPS
        return actual_watermark

    def forward(self, x, msg, global_step):
        num_samples = x.shape[-1]
        _, _, stft_result = self.stft.transform(x)
        # Evaluate how many chunks we can process
        # 2s input + 0.5s calculation delay
        # 2.00*16000 = 32000
        # 32800 // hop_length + 1 = 201 center=True
        # 0.5s*16000 = 8000
        # 8000 // hop_length + 1 = 51 center=True
        voice_prefilling = int((2.00*self.sampling_rate)//self.hop_length + 1)
        # Predict future 0.5s watermark
        # 0.5*16000 = 8000
        # 8000 // hop_length + 1 =51
        max_start = stft_result.shape[-1] - (voice_prefilling + self.delay_amt)
        if int(max_start / self.delay_amt) <= 0:
            return None  # Not enough frames for a chunk

        list_of_watermarks = []
        for i in range(int((stft_result.shape[-1] - (voice_prefilling + self.delay_amt)) / self.future_amt)):
            carrier_encoded = self.ENc(stft_result[:, :, :, i * self.future_amt:voice_prefilling + i * self.future_amt])
            # torch.Size([B, 1, 81])
            # torch.Size([B, 81, 1])
            # torch.Size([B, 1, 81, 1])
            # torch.Size([B, 1, 162, 201])
            watermark_encoded = self.msg_linear_in(msg).transpose(1, 2).unsqueeze(1).repeat(1, 1, 2,
                                                                                            carrier_encoded.shape[3])
            concatenated_feature = torch.cat((carrier_encoded, stft_result[:, :, :,
                                                               i*self.future_amt:voice_prefilling + i*self.future_amt], watermark_encoded), dim=1)
            # [B, 2, bins, length]
            # Embed the watermark
            carrier_watermarked = self.EM(concatenated_feature)
            # Append both the watermark chunk and the pilot segment
            list_of_watermarks.append(carrier_watermarked)

        if len(list_of_watermarks) > 0:
            watermark = torch.cat(list_of_watermarks, dim=-1)
            all_watermark_stft = self.pad_w_zero_stft(
                stft_result, watermark, voice_prefilling
            )
            del list_of_watermarks
            mask=stft_result!=0
            all_watermark_stft = all_watermark_stft*mask + 0.0000001

            self.stft.num_samples = num_samples

            # Recompute magnitude & phase
            real_part = all_watermark_stft[:, 0, :, :]
            imag_part = all_watermark_stft[:, 1, :, :]
            spect = torch.sqrt(real_part ** 2 + imag_part ** 2)
            phase = torch.atan2(imag_part, real_part)

            y = self.stft.inverse(spect, phase).squeeze(1)
            del spect, phase, real_part, imag_part

            with torch.no_grad():
                # Get chunk-level speech probabilities for the batch.
                # The output shape should be [batch, num_chunks]
                batch_chunk_probs = self.vad.audio_forward(x, sr=self.sampling_rate)

            # Threshold the probabilities to obtain a binary mask per chunk.
            batch_chunk_mask = (batch_chunk_probs > self.vad_threshold).float()

            # Upsample the chunk-level mask to a sample-level mask.
            # Each chunk's decision is repeated for chunk_size samples.
            sample_masks = torch.repeat_interleave(batch_chunk_mask, 512, dim=1).to(y.device)

            # Since the upsampled mask might be longer than the actual audio length,
            # slice the mask to match the original number of samples.
            sample_length = x.shape[-1]
            sample_masks = sample_masks[:, :sample_length]

            # Apply the mask to the original audio to zero out non-speech regions.
            masked_y = y * sample_masks
            return masked_y, all_watermark_stft
        else:
            print("Not enough watermarking!!!!")
            return None


class Decoder(nn.Module):
    def __init__(self, process_config, model_config, train_config, msg_length):
        super(Decoder, self).__init__()
        self.robust = model_config["robust"]
        # if self.robust:
        #     self.dl = distortion(process_config, train_config)
        self.mel_transform = TacotronSTFT(filter_length=process_config["mel"]["n_fft"], hop_length=process_config["mel"]["hop_length"], win_length=process_config["mel"]["win_length"])
        # self.vocoder = get_vocoder(device)
        self.vocoder_step = model_config["structure"]["vocoder_step"]
        self.win_dim = int((process_config["mel"]["n_fft"] / 2) + 1)
        self.hop_length = process_config["mel"]["hop_length"]
        self.block = model_config["conv2"]["block"]
        self.EX = WatermarkExtracter(input_channel=2, hidden_dim=model_config["conv2"]["hidden_dim"], block=self.block)
        self.stft = fixed_STFT(process_config["mel"]["n_fft"], process_config["mel"]["hop_length"], process_config["mel"]["win_length"])
        self.msg_linear_out = FCBlock(self.win_dim//2, msg_length, activation=LeakyReLU(inplace=True))

    def forward(self, y, global_step):
        y_identity = y
        # if global_step > self.vocoder_step:
        #     y_mel = self.mel_transform.mel_spectrogram(y.squeeze(1))
        #     # y = self.vocoder(y_mel)
        #     y_d = (self.mel_transform.griffin_lim(magnitudes=y_mel)).unsqueeze(1)
        # else:
        #     y_d = y
        y_d = y

        spect, phase, stft_result = self.stft.transform(y_d.squeeze(1))
        extracted_wm = self.EX(stft_result).squeeze(1)  # (B, win_dim, length)
        # Explicitly split the 162-dim vector into two halves of 81-dim each
        low, high = extracted_wm.chunk(2, dim=1)  # each has shape [B, win_dim / 2, length]
        low_msg = torch.mean(low, dim=2, keepdim=True).transpose(1,2)
        high_msg = torch.mean(high, dim=2, keepdim=True).transpose(1, 2)
        msg_avg = (low_msg + high_msg) / 2  # Average the two halves -> shape: [B, 1, 81]
        # msg = torch.mean(extracted_wm, dim=2, keepdim=True).transpose(1,2)
        # msg = self.msg_linear_out(msg)
        msg = self.msg_linear_out(msg_avg)

        _, _, stft_result_identity = self.stft.transform(y_identity)
        extracted_wm_identity = self.EX(stft_result_identity).squeeze(1)
        low_identity, high_identity = extracted_wm_identity.chunk(2, dim=1)  # each has shape [B, win_dim / 2, length]
        low_msg_identity = torch.mean(low_identity, dim=2, keepdim=True).transpose(1, 2)
        high_msg_identity = torch.mean(high_identity, dim=2, keepdim=True).transpose(1, 2)
        msg_avg_identity = (low_msg_identity + high_msg_identity) / 2  # Average the two halves -> shape: [B, 1, 81]
        # msg_identity = torch.mean(extracted_wm_identity,dim=2, keepdim=True).transpose(1,2)
        # msg_identity = self.msg_linear_out(msg_identity)
        msg_identity = self.msg_linear_out(msg_avg_identity)
        del stft_result, stft_result_identity, extracted_wm, extracted_wm_identity
        return msg, msg_identity

class Discriminator(nn.Module):
    def __init__(self, process_config):
        super(Discriminator, self).__init__()
        self.conv = nn.Sequential(
                ReluBlock(2,16,3,1,1),
                ReluBlock(16,32,3,1,1),
                ReluBlock(32,64,3,1,1),
                nn.AdaptiveAvgPool2d(output_size=(1, 1))
                )
        self.linear = nn.Linear(64,1)
        self.stft = fixed_STFT(process_config["mel"]["n_fft"], process_config["mel"]["hop_length"], process_config["mel"]["win_length"])

    def forward(self, x):
        _, _, stft_result = self.stft.transform(x)
        x = self.conv(stft_result)
        x = x.squeeze(2).squeeze(2)
        x = self.linear(x)
        return x

In [28]:
process_config = yaml.load(open("./config/process.yaml", "r"), Loader=yaml.FullLoader)
model_config = yaml.load(open("./config/model.yaml", "r"), Loader=yaml.FullLoader)
train_config = yaml.load(open("./config/train.yaml", "r"), Loader=yaml.FullLoader)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

encoder = Encoder(process_config, model_config, train_config, train_config["watermark"]["length"]).to(device)
decoder = Decoder(process_config, model_config, train_config, train_config["watermark"]["length"]).to(device)
discriminator = Discriminator(process_config).to(device)
checkpoint_path = "ablation/original_baseline_split_frequency_vad_none-conv2_ep_30_2025-05-03_20_37_29.pth.tar"
checkpoint = torch.load(checkpoint_path, map_location=device)
encoder.load_state_dict(checkpoint["encoder"])
decoder.load_state_dict(checkpoint["decoder"])

<All keys matched successfully>

## Single Inference

In [29]:
with torch.inference_mode():
    encoder.eval()
    decoder.eval()
    msg = generate_random_msg(1, train_config["watermark"]["length"], device)
    # wav, sr = torchaudio.load("ablation/260-123440-0016.wav")
    # wav, sr = torchaudio.load("ablation/LJ001-0052.wav")
    wav, sr = torchaudio.load("ablation/LJ001-0053.wav")
    wav = wav.to(device) # (batch, length)
    watermark, carrier_wateramrked = encoder(wav, msg, 1) # (batch, 1, length)
    y_wm = wav + watermark
    decoded = decoder(y_wm, 1)
    decoder_acc = [((decoded[0] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item(),
                           ((decoded[1] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item()]
    print("Accuracy:[{:.8f},{:.8f}]".format(*decoder_acc))
    print("original msg:", msg)
    print("decoded msg:", decoded[0])

    # y_norm = normalize_audio(wav.cpu().detach())
    wm_norm = normalize_audio(watermark[0].cpu().detach())
    y_wm_norm = normalize_audio(y_wm[0].cpu().detach())

    # save_spectrogram(y_norm.squeeze(0), "ablation/orig_spectrogram.png")
    save_spectrogram(wm_norm, "ablation/MSE_loudness_split_frequency_vad_watermark_spectrogram.png")
    save_spectrogram(y_wm_norm, "ablation/MSE_loudness_split_vad_frequency_watermarked_spectrogram.png")

    # save_audio(wav, "ablation/original_audio.wav", sample_rate=sr)
    save_audio(watermark, "ablation/MSE_loudness_split_frequency_vad_watermark_audio.wav", sample_rate=sr)
    save_audio(y_wm, "ablation/MSE_loudness_split_frequency_vad_watermarked_audio.wav", sample_rate=sr)

Accuracy:[1.00000000,1.00000000]
original msg: tensor([[[-1.,  1.,  1., -1.,  1.,  1.,  1.,  1.,  1., -1.]]], device='cuda:0')
decoded msg: tensor([[[-0.0116,  1.1637,  1.1584, -0.0140,  1.2394,  1.4645,  1.1200,
           1.2094,  1.2667, -0.0166]]], device='cuda:0')


### Original Audio


In [22]:
Audio(wav.cpu(), rate=sr)

### Watermarked Audio

In [23]:
Audio(y_wm.cpu(), rate=sr)

## Batch Inference

### LibriSpeech

In [ ]:
train_config["path"]["raw_path"] = "/home/yizhu/Data/LibriSpeech_wav"
dev_audios = MyDataset(
    process_config=process_config, train_config=train_config, flag="test"
)
dev_audios_loader = DataLoader(
        dev_audios,
        batch_size=2,
        shuffle=False,
        collate_fn=collate_fn,
        pin_memory=True,
        num_workers=20,
        persistent_workers=True,
    )

In [ ]:
with torch.inference_mode():
    encoder.eval()
    decoder.eval()
    discriminator.eval()
    test_avg_acc = [0, 0]
    test_avg_snr = 0
    count = 0
    for sample in track(dev_audios_loader):
        count += 1
        b = sample["matrix"].shape[0]
        # ---------------- build watermark
        wav_matrix = sample["matrix"].to(device)
        msg = generate_random_msg(wav_matrix.size(0), train_config["watermark"]["length"], device)
        watermark, carrier_wateramrked = encoder(wav_matrix, msg, 1)
        y_wm = wav_matrix + watermark
        decoded = decoder(y_wm, 1)
        decoder_acc = [((decoded[0] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item(),
                           ((decoded[1] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item()]
        zero_tensor = torch.zeros(wav_matrix.shape).to(device)
        snr = 10 * torch.log10(
            mse_loss(wav_matrix.detach(), zero_tensor) / mse_loss(wav_matrix.detach(), y_wm.detach()))

        test_avg_snr += snr
        test_avg_acc[0] += decoder_acc[0]
        test_avg_acc[1] += decoder_acc[1]

    test_avg_acc[0] /= count
    test_avg_acc[1] /= count
    test_avg_snr /= count
    print("Test Average SNR:", test_avg_snr)
    print("Test Average Accuracy:", test_avg_acc[0])

In [ ]:
count

### LJSpeech

In [ ]:
train_config["path"]["raw_path"] = "/home/yizhu/Data/LJSpeech-1.1"
dev_audios = MyDataset(
    process_config=process_config, train_config=train_config, flag="test"
)
dev_audios_loader = DataLoader(
        dev_audios,
        batch_size=2,
        shuffle=False,
        collate_fn=collate_fn,
        pin_memory=True,
        num_workers=20,
        persistent_workers=True,
    )

In [ ]:
with torch.inference_mode():
    encoder.eval()
    decoder.eval()
    discriminator.eval()
    test_avg_acc = [0, 0]
    test_avg_snr = 0
    count = 0
    for sample in track(dev_audios_loader):
        count += 1
        b = sample["matrix"].shape[0]
        # ---------------- build watermark
        wav_matrix = sample["matrix"].to(device)
        msg = generate_random_msg(wav_matrix.size(0), train_config["watermark"]["length"], device)
        watermark, carrier_wateramrked = encoder(wav_matrix, msg, 1)
        y_wm = wav_matrix + watermark
        decoded = decoder(y_wm, 1)
        decoder_acc = [((decoded[0] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item(),
                           ((decoded[1] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item()]
        zero_tensor = torch.zeros(wav_matrix.shape).to(device)
        snr = 10 * torch.log10(
            mse_loss(wav_matrix.detach(), zero_tensor) / mse_loss(wav_matrix.detach(), y_wm_aug.detach()))

        test_avg_snr += snr
        test_avg_acc[0] += decoder_acc[0]
        test_avg_acc[1] += decoder_acc[1]

    test_avg_acc[0] /= count
    test_avg_acc[1] /= count
    test_avg_snr /= count
    print("Test Average SNR:", test_avg_snr)
    print("Test Average Accuracy:", test_avg_acc[0])

## Background Noise

### 3-30 snr db

#### LibriSpeech

In [ ]:
train_config["path"]["raw_path"] = "/home/yizhu/Data/LibriSpeech_wav"
dev_audios = MyDataset(
    process_config=process_config, train_config=train_config, flag="test"
)
dev_audios_loader = DataLoader(
        dev_audios,
        batch_size=2,
        shuffle=False,
        collate_fn=collate_fn,
        pin_memory=True,
        num_workers=20,
        persistent_workers=True,
    )

In [ ]:
from audiomentations import AddBackgroundNoise, PolarityInversion

# Fix randomness
random.seed(42)
np.random.seed(42)

transform = AddBackgroundNoise(
    sounds_path="ablation/ESC-50-master/audio",  # Replace with real path
    min_snr_db=3.0,
    max_snr_db=30.0,
    noise_transform=PolarityInversion(),
    p=1.0
)

with torch.inference_mode():
    encoder.eval()
    decoder.eval()
    discriminator.eval()
    test_avg_acc = [0, 0]
    test_avg_snr = 0
    count = 0
    for sample in track(dev_audios_loader):
        count += 1
        b = sample["matrix"].shape[0]
        # ---------------- build watermark
        wav_matrix = sample["matrix"].to(device)
        # Fixed message for testing (same for all in batch)
        msg = torch.tensor([-1.,  1.,  1., -1., -1.,  1., -1.,  1., -1., -1.], dtype=torch.float32)
        msg = msg.unsqueeze(0).repeat(b, 1, 1).to(device)

        # Watermark embedding
        watermark, carrier_wateramrked = encoder(wav_matrix, msg, 1)
        y_wm = wav_matrix + watermark

        # Convert each sample to NumPy, apply random noise, convert back
        # Convert each sample to NumPy, apply random noise, convert back
        augmented_waveforms = []
        for i in range(y_wm.shape[0]):
            waveform_np = y_wm[i].detach().cpu().numpy()
            if waveform_np.ndim > 1:
                waveform_np = waveform_np[0]  # Select first channel if stereo
            augmented_np = transform(samples=waveform_np, sample_rate=16000)
            augmented_waveforms.append(torch.tensor(augmented_np).unsqueeze(0))  # Add (1, L)

        y_wm_aug = torch.cat(augmented_waveforms, dim=0).to(device)  # Shape: (B, L)

        decoded = decoder(y_wm_aug, 1)
        decoder_acc = [((decoded[0] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item(),
                           ((decoded[1] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item()]
        zero_tensor = torch.zeros(wav_matrix.shape).to(device)
        snr = 10 * torch.log10(
            mse_loss(wav_matrix.detach(), zero_tensor) / mse_loss(wav_matrix.detach(), y_wm_aug.detach()))

        test_avg_snr += snr
        test_avg_acc[0] += decoder_acc[0]
        test_avg_acc[1] += decoder_acc[1]

    test_avg_acc[0] /= count
    test_avg_acc[1] /= count
    test_avg_snr /= count
    print("Test Average SNR:", test_avg_snr)
    print("Test Average Accuracy:", test_avg_acc[0])

#### LJSpeech

In [ ]:
train_config["path"]["raw_path"] = "/home/yizhu/Data/LJSpeech-1.1"
dev_audios = MyDataset(
    process_config=process_config, train_config=train_config, flag="test"
)
dev_audios_loader = DataLoader(
        dev_audios,
        batch_size=2,
        shuffle=False,
        collate_fn=collate_fn,
        pin_memory=True,
        num_workers=20,
        persistent_workers=True,
    )

In [ ]:
from audiomentations import AddBackgroundNoise, PolarityInversion

# Fix randomness
random.seed(42)
np.random.seed(42)

transform = AddBackgroundNoise(
    sounds_path="ablation/ESC-50-master/audio",  # Replace with real path
    min_snr_db=3.0,
    max_snr_db=30.0,
    noise_transform=PolarityInversion(),
    p=1.0
)

with torch.inference_mode():
    encoder.eval()
    decoder.eval()
    discriminator.eval()
    test_avg_acc = [0, 0]
    test_avg_snr = 0
    count = 0
    for sample in track(dev_audios_loader):
        count += 1
        b = sample["matrix"].shape[0]
        # ---------------- build watermark
        wav_matrix = sample["matrix"].to(device)
        # Fixed message for testing (same for all in batch)
        msg = torch.tensor([-1.,  1.,  1., -1., -1.,  1., -1.,  1., -1., -1.], dtype=torch.float32)
        msg = msg.unsqueeze(0).repeat(b, 1, 1).to(device)

        # Watermark embedding
        watermark, carrier_wateramrked = encoder(wav_matrix, msg, 1)
        y_wm = wav_matrix + watermark

        # Convert each sample to NumPy, apply random noise, convert back
        # Convert each sample to NumPy, apply random noise, convert back
        augmented_waveforms = []
        for i in range(y_wm.shape[0]):
            waveform_np = y_wm[i].detach().cpu().numpy()
            if waveform_np.ndim > 1:
                waveform_np = waveform_np[0]  # Select first channel if stereo
            augmented_np = transform(samples=waveform_np, sample_rate=16000)
            augmented_waveforms.append(torch.tensor(augmented_np).unsqueeze(0))  # Add (1, L)

        y_wm_aug = torch.cat(augmented_waveforms, dim=0).to(device)  # Shape: (B, L)

        decoded = decoder(y_wm_aug, 1)
        decoder_acc = [((decoded[0] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item(),
                           ((decoded[1] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item()]
        zero_tensor = torch.zeros(wav_matrix.shape).to(device)
        snr = 10 * torch.log10(
            mse_loss(wav_matrix.detach(), zero_tensor) / mse_loss(wav_matrix.detach(), y_wm_aug.detach()))

        test_avg_snr += snr
        test_avg_acc[0] += decoder_acc[0]
        test_avg_acc[1] += decoder_acc[1]

    test_avg_acc[0] /= count
    test_avg_acc[1] /= count
    test_avg_snr /= count
    print("Test Average SNR:", test_avg_snr)
    print("Test Average Accuracy:", test_avg_acc[0])

### 5 snr db

#### LibriSpeech

In [ ]:
train_config["path"]["raw_path"] = "/home/yizhu/Data/LibriSpeech_wav"
dev_audios = MyDataset(
    process_config=process_config, train_config=train_config, flag="test"
)
dev_audios_loader = DataLoader(
        dev_audios,
        batch_size=2,
        shuffle=False,
        collate_fn=collate_fn,
        pin_memory=True,
        num_workers=20,
        persistent_workers=True,
    )

In [ ]:
from audiomentations import AddBackgroundNoise, PolarityInversion

# Fix randomness
random.seed(42)
np.random.seed(42)

transform = AddBackgroundNoise(
    sounds_path="ablation/ESC-50-master/audio",  # Replace with real path
    min_snr_db=5.0,
    max_snr_db=5.0,
    noise_transform=PolarityInversion(),
    p=1.0
)

with torch.inference_mode():
    encoder.eval()
    decoder.eval()
    discriminator.eval()
    test_avg_acc = [0, 0]
    test_avg_snr = 0
    count = 0
    for sample in track(dev_audios_loader):
        count += 1
        b = sample["matrix"].shape[0]
        # ---------------- build watermark
        wav_matrix = sample["matrix"].to(device)
        # Fixed message for testing (same for all in batch)
        msg = torch.tensor([-1.,  1.,  1., -1., -1.,  1., -1.,  1., -1., -1.], dtype=torch.float32)
        msg = msg.unsqueeze(0).repeat(b, 1, 1).to(device)

        # Watermark embedding
        watermark, carrier_wateramrked = encoder(wav_matrix, msg, 1)
        y_wm = wav_matrix + watermark

        # Convert each sample to NumPy, apply random noise, convert back
        # Convert each sample to NumPy, apply random noise, convert back
        augmented_waveforms = []
        for i in range(y_wm.shape[0]):
            waveform_np = y_wm[i].detach().cpu().numpy()
            if waveform_np.ndim > 1:
                waveform_np = waveform_np[0]  # Select first channel if stereo
            augmented_np = transform(samples=waveform_np, sample_rate=16000)
            augmented_waveforms.append(torch.tensor(augmented_np).unsqueeze(0))  # Add (1, L)

        y_wm_aug = torch.cat(augmented_waveforms, dim=0).to(device)  # Shape: (B, L)

        decoded = decoder(y_wm_aug, 1)
        decoder_acc = [((decoded[0] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item(),
                           ((decoded[1] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item()]
        zero_tensor = torch.zeros(wav_matrix.shape).to(device)
        snr = 10 * torch.log10(
            mse_loss(wav_matrix.detach(), zero_tensor) / mse_loss(wav_matrix.detach(), y_wm_aug.detach()))

        test_avg_snr += snr
        test_avg_acc[0] += decoder_acc[0]
        test_avg_acc[1] += decoder_acc[1]

    test_avg_acc[0] /= count
    test_avg_acc[1] /= count
    test_avg_snr /= count
    print("Test Average SNR:", test_avg_snr)
    print("Test Average Accuracy:", test_avg_acc[0])

#### LJSpeech

In [ ]:
train_config["path"]["raw_path"] = "/home/yizhu/Data/LJSpeech-1.1"
dev_audios = MyDataset(
    process_config=process_config, train_config=train_config, flag="test"
)
dev_audios_loader = DataLoader(
        dev_audios,
        batch_size=2,
        shuffle=False,
        collate_fn=collate_fn,
        pin_memory=True,
        num_workers=20,
        persistent_workers=True,
    )

In [ ]:
from audiomentations import AddBackgroundNoise, PolarityInversion

# Fix randomness
random.seed(42)
np.random.seed(42)

transform = AddBackgroundNoise(
    sounds_path="ablation/ESC-50-master/audio",  # Replace with real path
    min_snr_db=5.0,
    max_snr_db=5.0,
    noise_transform=PolarityInversion(),
    p=1.0
)

with torch.inference_mode():
    encoder.eval()
    decoder.eval()
    discriminator.eval()
    test_avg_acc = [0, 0]
    test_avg_snr = 0
    count = 0
    for sample in track(dev_audios_loader):
        count += 1
        b = sample["matrix"].shape[0]
        # ---------------- build watermark
        wav_matrix = sample["matrix"].to(device)
        # Fixed message for testing (same for all in batch)
        msg = torch.tensor([-1.,  1.,  1., -1., -1.,  1., -1.,  1., -1., -1.], dtype=torch.float32)
        msg = msg.unsqueeze(0).repeat(b, 1, 1).to(device)

        # Watermark embedding
        watermark, carrier_wateramrked = encoder(wav_matrix, msg, 1)
        y_wm = wav_matrix + watermark

        # Convert each sample to NumPy, apply random noise, convert back
        # Convert each sample to NumPy, apply random noise, convert back
        augmented_waveforms = []
        for i in range(y_wm.shape[0]):
            waveform_np = y_wm[i].detach().cpu().numpy()
            if waveform_np.ndim > 1:
                waveform_np = waveform_np[0]  # Select first channel if stereo
            augmented_np = transform(samples=waveform_np, sample_rate=16000)
            augmented_waveforms.append(torch.tensor(augmented_np).unsqueeze(0))  # Add (1, L)

        y_wm_aug = torch.cat(augmented_waveforms, dim=0).to(device)  # Shape: (B, L)

        decoded = decoder(y_wm_aug, 1)
        decoder_acc = [((decoded[0] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item(),
                           ((decoded[1] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item()]
        zero_tensor = torch.zeros(wav_matrix.shape).to(device)
        snr = 10 * torch.log10(
            mse_loss(wav_matrix.detach(), zero_tensor) / mse_loss(wav_matrix.detach(), y_wm_aug.detach()))

        test_avg_snr += snr
        test_avg_acc[0] += decoder_acc[0]
        test_avg_acc[1] += decoder_acc[1]

    test_avg_acc[0] /= count
    test_avg_acc[1] /= count
    test_avg_snr /= count
    print("Test Average SNR:", test_avg_snr)
    print("Test Average Accuracy:", test_avg_acc[0])

## Editing

### Normal Editing

#### LibriSpeech

In [ ]:
irrelevant_paths = glob("/home/yizhu/Data/LJSpeech-1.1/test/*.wav")
train_config["path"]["raw_path"] = "/home/yizhu/Data/LibriSpeech_wav"
dev_audios = MyDataset(
    process_config=process_config, train_config=train_config, flag="test"
)
dev_audios_loader = DataLoader(
        dev_audios,
        batch_size=2,
        shuffle=False,
        collate_fn=collate_fn,
        pin_memory=True,
        num_workers=20,
        persistent_workers=True,
    )

In [ ]:
with torch.inference_mode():
    encoder.eval()
    decoder.eval()
    discriminator.eval()
    test_avg_acc = [0, 0]
    test_avg_snr = 0
    count = 0
    for sample in track(dev_audios_loader):
        count += 1
        b = sample["matrix"].shape[0]
        # ---------------- build watermark
        wav_matrix = sample["matrix"].to(device)
        msg = generate_random_msg(wav_matrix.size(0), train_config["watermark"]["length"], device)
        watermark, carrier_wateramrked = encoder(wav_matrix, msg, 1)
        y_wm = wav_matrix + watermark

        y_wm_aug = splice_with_irrelevant(y_wm.unsqueeze(1), irrelevant_paths, sample_rate=16000)

        decoded = decoder(y_wm_aug.squeeze(1), 1)
        decoder_acc = [((decoded[0] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item(),
                           ((decoded[1] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item()]
        zero_tensor = torch.zeros(wav_matrix.shape).to(device)
        snr = 10 * torch.log10(
            mse_loss(wav_matrix.detach(), zero_tensor) / mse_loss(wav_matrix.detach(), y_wm_aug.detach()))

        test_avg_snr += snr
        test_avg_acc[0] += decoder_acc[0]
        test_avg_acc[1] += decoder_acc[1]

    test_avg_acc[0] /= count
    test_avg_acc[1] /= count
    test_avg_snr /= count
    print("Test Average SNR:", test_avg_snr)
    print("Test Average Accuracy:", test_avg_acc[0])

#### LJSpeech

In [ ]:
irrelevant_paths = glob("/home/yizhu/Data/LibriSpeech_wav/test_orig/*.wav")
train_config["path"]["raw_path"] = "/home/yizhu/Data/LJSpeech-1.1"
dev_audios = MyDataset(
    process_config=process_config, train_config=train_config, flag="test"
)
dev_audios_loader = DataLoader(
        dev_audios,
        batch_size=2,
        shuffle=False,
        collate_fn=collate_fn,
        pin_memory=True,
        num_workers=20,
        persistent_workers=True,
    )

In [ ]:
with torch.inference_mode():
    encoder.eval()
    decoder.eval()
    discriminator.eval()
    test_avg_acc = [0, 0]
    test_avg_snr = 0
    count = 0
    for sample in track(dev_audios_loader):
        count += 1
        b = sample["matrix"].shape[0]
        # ---------------- build watermark
        wav_matrix = sample["matrix"].to(device)
        msg = generate_random_msg(wav_matrix.size(0), train_config["watermark"]["length"], device)
        watermark, carrier_wateramrked = encoder(wav_matrix, msg, 1)
        y_wm = wav_matrix + watermark

        y_wm_aug = splice_with_irrelevant(y_wm.unsqueeze(1), irrelevant_paths, sample_rate=16000)

        decoded = decoder(y_wm_aug.squeeze(1), 1)
        decoder_acc = [((decoded[0] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item(),
                           ((decoded[1] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item()]
        zero_tensor = torch.zeros(wav_matrix.shape).to(device)
        snr = 10 * torch.log10(
            mse_loss(wav_matrix.detach(), zero_tensor) / mse_loss(wav_matrix.detach(), y_wm_aug.detach()))

        test_avg_snr += snr
        test_avg_acc[0] += decoder_acc[0]
        test_avg_acc[1] += decoder_acc[1]

    test_avg_acc[0] /= count
    test_avg_acc[1] /= count
    test_avg_snr /= count
    print("Test Average SNR:", test_avg_snr)
    print("Test Average Accuracy:", test_avg_acc[0])

### Extreme Editing

#### LibriSpeech

In [25]:
irrelevant_paths = glob("/home/yizhu/Data/LJSpeech-1.1/test/*.wav")
train_config["path"]["raw_path"] = "/home/yizhu/Data/LibriSpeech_wav"
process_config["audio"]["or_sample_rate"]  = 16000
dev_audios = MyDataset(
    process_config=process_config, train_config=train_config, flag="test"
)
dev_audios_loader = DataLoader(
        dev_audios,
        batch_size=2,
        shuffle=False,
        collate_fn=collate_fn,
        pin_memory=True,
        num_workers=20,
        persistent_workers=True,
    )

In [26]:
with torch.inference_mode():
    encoder.eval()
    decoder.eval()
    discriminator.eval()
    test_avg_acc = [0, 0]
    test_avg_snr = 0
    count = 0
    for sample in track(dev_audios_loader):
        count += 1
        b = sample["matrix"].shape[0]
        # ---------------- build watermark
        wav_matrix = sample["matrix"].to(device)
        msg = generate_random_msg(wav_matrix.size(0), train_config["watermark"]["length"], device)
        watermark, carrier_wateramrked = encoder(wav_matrix, msg, 1)
        y_wm = wav_matrix + watermark

        y_wm_aug = splice_with_irrelevant_scattered(y_wm.unsqueeze(1), irrelevant_paths, sample_rate=16000, num_segments=5)

        decoded = decoder(y_wm_aug.squeeze(1), 1)
        decoder_acc = [((decoded[0] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item(),
                           ((decoded[1] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item()]
        zero_tensor = torch.zeros(wav_matrix.shape).to(device)
        snr = 10 * torch.log10(
            mse_loss(wav_matrix.detach(), zero_tensor) / mse_loss(wav_matrix.detach(), y_wm_aug.detach()))

        test_avg_snr += snr
        test_avg_acc[0] += decoder_acc[0]
        test_avg_acc[1] += decoder_acc[1]

    test_avg_acc[0] /= count
    test_avg_acc[1] /= count
    test_avg_snr /= count
    print("Test Average SNR:", test_avg_snr)
    print("Test Average Accuracy:", test_avg_acc[0])

Test Average SNR: tensor(-3.5565, device='cuda:0')
Test Average Accuracy: 0.8635426084556922


#### LJSpeech

In [27]:
irrelevant_paths = glob("/home/yizhu/Data/LibriSpeech_wav/test/*.wav")
train_config["path"]["raw_path"] = "/home/yizhu/Data/LJSpeech-1.1"
process_config["audio"]["or_sample_rate"]  = 22050
dev_audios = MyDataset(
    process_config=process_config, train_config=train_config, flag="test"
)
dev_audios_loader = DataLoader(
        dev_audios,
        batch_size=2,
        shuffle=False,
        collate_fn=collate_fn,
        pin_memory=True,
        num_workers=20,
        persistent_workers=True,
    )

In [28]:
with torch.inference_mode():
    encoder.eval()
    decoder.eval()
    discriminator.eval()
    test_avg_acc = [0, 0]
    test_avg_snr = 0
    count = 0
    for sample in track(dev_audios_loader):
        count += 1
        b = sample["matrix"].shape[0]
        # ---------------- build watermark
        wav_matrix = sample["matrix"].to(device)
        msg = generate_random_msg(wav_matrix.size(0), train_config["watermark"]["length"], device)
        watermark, carrier_wateramrked = encoder(wav_matrix, msg, 1)
        y_wm = wav_matrix + watermark

        y_wm_aug = splice_with_irrelevant_scattered(y_wm.unsqueeze(1), irrelevant_paths, sample_rate=16000, num_segments=5)

        decoded = decoder(y_wm_aug.squeeze(1), 1)
        decoder_acc = [((decoded[0] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item(),
                           ((decoded[1] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item()]
        zero_tensor = torch.zeros(wav_matrix.shape).to(device)
        snr = 10 * torch.log10(
            mse_loss(wav_matrix.detach(), zero_tensor) / mse_loss(wav_matrix.detach(), y_wm_aug.detach()))

        test_avg_snr += snr
        test_avg_acc[0] += decoder_acc[0]
        test_avg_acc[1] += decoder_acc[1]

    test_avg_acc[0] /= count
    test_avg_acc[1] /= count
    test_avg_snr /= count
    print("Test Average SNR:", test_avg_snr)
    print("Test Average Accuracy:", test_avg_acc[0])

Test Average SNR: tensor(-1.9750, device='cuda:0')
Test Average Accuracy: 0.9278060434527345


In [29]:
Audio(y_wm[0].cpu(), rate=16000)

In [30]:
Audio(y_wm_aug[0].cpu(), rate=16000)